# ACTION 2026 — Data Mining, Babak Final: Klasifikasi Sentimen 4 Kelas pada 9 Pilar Layanan Publik
**Tim sistemhitam** — Tamam Fajar Briliansyah, Hamasah Fatiy Dakhilullah, Moh. Rizky Rahmadian Makkani

Setiap komentar diberi satu dari 4 kelas (`TidakRelevan` / `Negatif` / `Netral` / `Positif`) untuk masing-masing 9 pilar. Metrik: *Mean Column-wise Macro-F1* atas 4 kelas.

**Temuan kunci.** Train dan test final identik baris-per-baris dengan babak penyisihan; hanya labelnya yang berubah: setiap sel yang dulu `Relevan` kini dipecah menjadi `Negatif`/`Netral`/`Positif`. Sentimen melekat pada komentar (pada 92,6% baris multi-label sentimennya sama di semua pilarnya). Karena itu masalah dipecah menjadi dua tahap:

1. **Relevansi per pilar** — prediksi stacker babak penyisihan (folder `artifacts_final/penyisihan`, CV 0,937) dipakai ulang apa adanya.
2. **Sentimen per komentar** — model baru yang dilatih di notebook ini: TF-IDF (cadangan), IndoBERT-large, dan Sailor2-3B (LoRA), keluaran 3 sigmoid `[Negatif, Netral, Positif]`.
3. **Perakitan** — sel = `TidakRelevan` bila P(relevan) < threshold v9; selain itu `argmax(log p_sentimen + bias kelas)`. Bias kelas dan kombinasi model dipilih lewat CV (bias disetel di fold lain).

**Cara menjalankan.** Letakkan notebook ini sejajar dengan folder `action-2026-data-mining-babak-final/` (train/test/sample_submission final) dan `artifacts_final/` (subfolder `penyisihan/` berisi artefak babak penyisihan dan wajib ada; sisanya hasil pelatihan notebook ini), lalu *Run All*. Setiap model yang selesai langsung menghasilkan `submission.csv` yang diperbarui; model yang sudah selesai dimuat dari `artifacts_final/` saat notebook dijalankan ulang. Tidak ada data eksternal dan tidak ada pemanggilan API LLM.

## 0. Setup & konfigurasi

In [1]:
import os, sys, gc, json, math, time, random, glob, itertools, traceback, warnings, platform
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn
import sklearn, transformers
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging

warnings.filterwarnings("ignore"); hf_logging.set_verbosity_error()
pd.set_option("display.width", 200)

DATA_DIR = "data"      # train.csv, test.csv, sample_submission.csv babak final
V9 = "artifacts_final/penyisihan"                     # artefak babak penyisihan (relevansi per pilar, fold, kosakata normalizer)
ART = "artifacts_final"                               # artefak notebook ini (model sentimen)
IBL_NAME = os.environ.get("IBL_NAME", "indobenchmark/indobert-large-p2")
LLM_NAME = os.environ.get("LLM_NAME", "sail/Sailor2-3B")
SEED, NOUT, REUSE_L1 = 42, 3, True
LABELS = ["Infrastruktur", "Kesehatan", "Administrasi_Perizinan", "Lingkungan_Kebersihan", "Ketertiban_Umum",
          "Transportasi_LaluLintas", "Sosial_Bantuan", "Keamanan_Kriminalitas", "Pendidikan"]
SHORT = ["Infra", "Kes", "Adm", "Ling", "Tertib", "Trans", "Sos", "Keam", "Pend"]
SENT = np.array(["Negatif", "Netral", "Positif"])
C4 = ["TidakRelevan", "Negatif", "Netral", "Positif"]
EPS = 1e-4
for _d in [ART, f"{ART}/l1", f"{ART}/models"]: os.makedirs(_d, exist_ok=True)

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
if not torch.cuda.is_available():                     # tanpa GPU hanya bagian CPU (TF-IDF & perakitan) yang bisa berjalan
    torch.cuda.reset_peak_memory_stats = lambda *a, **k: None; torch.cuda.max_memory_reserved = lambda *a, **k: 0; torch.cuda.empty_cache = lambda *a, **k: None
print("python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(tidak ada GPU)")

C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python 3.12.10 | torch 2.11.0+cu128 | transformers 4.57.6 | sklearn 1.7.2
CUDA: True NVIDIA GeForce RTX 5060 Ti


## 1. Normalizer de-obfuscation
Modul `textnorm.py` yang sama dengan babak penyisihan (teksnya identik, termasuk noise homoglyph, leetspeak, huruf berspasi, kata tergeser). Kosakatanya dimuat dari `artifacts_final/penyisihan/models/textnorm_vocab.joblib`.

In [2]:
%%writefile textnorm.py
"""
textnorm.py - robust de-obfuscating text normalizer for the ACTION 2026 complaint data.

Public API
----------
build_vocab(train_texts, extra_words=None, min_freq=2) -> Vocab   (dict[str,int] subclass, word -> train freq)
Normalizer(vocab, lowercase=True)                      -> callable  str -> str  (internally cached)
normalize_series(series, normalizer)                    -> pd.Series (maps unique values only, NaN -> '')
load_default_extra_words()                              -> set[str]  whole-word tokens of indobert-base-p1 vocab
default_normalizer(train_csv)                           -> (vocab, Normalizer) built exactly as in production

Typical use
-----------
    import textnorm as T
    vocab = T.build_vocab(list(train.Teks) + list(train.Judul_Caption.dropna().unique()),
                          T.load_default_extra_words())
    norm = T.Normalizer(vocab)                 # lowercase output; Normalizer(vocab, lowercase=False) keeps case
    train['teks_norm'] = T.normalize_series(train.Teks, norm)

Pipeline (per text)
-------------------
 1. NaN-safe; NFKC (fullwidth / math-alphanumerics -> ASCII, '²'->'2'); delete zero-width / invisible chars
    (U+200B-U+200F, U+2060.., U+FEFF, variation selectors...). In this data they NEVER sit between two real
    words (checked), so deletion cannot merge words. Braille blank U+2800, hangul fillers and exotic spaces
    (U+2002...) become ' '. Combining marks on Latin letters are stripped.
 2. Homoglyphs (Cyrillic / Greek / small capitals) -> Latin, both cases.
 3. URL -> 'url', @mention -> '@user' (protected placeholders, never re-normalized). Bare domains need clean
    dot-separated labels and a non-'.org' TLD ('goblok...org' / 'y.org' = '... orang', not URLs).
 4. Letter-separated obfuscation: dotted 'p.a.r.k.i.r. .l.i.a.r' / 'B.P.J.S' and spaced 'b e g a l' /
    'J a l a n R u s a k' (runs of >=3 single letters) are joined; multi-word runs are re-segmented with a
    bigram-aware vocab DP (train word/bigram counts, OOV pieces via char LM, camel-case boundaries get a bonus).
 5. Hashtags: '#' stripped, leet/caesar-decoded, camel split and vocab-DP split ('#jalanrusak' -> 'jalan rusak');
    an upper-case run before lower case is resolved with the vocab ('#SARuntuksemua' -> 'sar untuk semua') and an
    unknown camel piece gluing frequent words is split ('#TuntasdanBangkit' -> 'tuntas dan bangkit').
 6. Per token (cached by surface form):
      - numbers, money (Rp..), counts with units (10rb, 3x, 2jt), codes (bos120), known alnum acronyms kept
      - reduplication 'anak2' -> 'anak-anak', 'mudah2an' -> 'mudah-mudahan'; prefixed bases reduplicate the root
        ('bertahun2' -> 'bertahun-tahun', 'sebaik2nya' -> 'sebaik-baiknya'; hand-written list + forms written with a
        hyphen in TRAIN text + 'ber'+unit / vocab-frequency rule); 'gara2x' -> 'gara-gara'; 'ke2' -> 'kedua';
        a glued word is split off ('toko2itu' -> 'toko-toko itu'); depth-limited (no recursion blow-up)
      - leetspeak (injector map a/e/i/o/s/g -> 4/3/1/0/5/9; 1 may also be 'l'), vocab-validated; tokens with
        none of the letters a,e,i,o,s,g that still contain those digits are decoded even if OOV; leet on top of a
        Caesar shift is decoded like a plain Caesar word ('03n3hm3r' -> 'kajadian'), and overrides the code-like
        guard only with strong evidence ('umzm51' -> 'mereka', 'xcv91' -> 'punya'); Roman numerals kept ('VIII')
      - elongation 'paraaahhh' -> 'parah', 'Hayoooo' -> 'hayo' (vocab-validated; word-initial runs kept: PPPK)
      - per-word Caesar decoding. Measured on train: the injector shifts ONLY words of length >= 4, with shifts
        {3,4,5,7,8,9,11,13}, ~9.8% of eligible words, and NEVER the topic keywords (jalan, sekolah, parkir, ...).
        Noisy-channel decision: argmax over {identity} U {shift(w,-k)} of
            log P_unigram(candidate) + log P_channel(k | candidate)
        where P_channel uses a per-word Bayesian shift-rate (keywords ~0) and OOV strings get
        P = oov_mass * P_charLM (char-trigram LM on train + pretrained-tokenizer word types).
      - small hand-written slang / abbreviation dictionary
 0. (optional, default on) the injected trailing emoji (label-independent noise on ~51% of rows) is dropped.
 7. Emoji / punctuation repeats collapsed (>3 -> 3), whitespace collapsed, lowercased (option keep case).

Only TRAIN text (+ the pretrained tokenizer vocabulary) is used to build the vocab; nothing is fit on test.
"""
from __future__ import annotations

import math
import re
import unicodedata
from collections import Counter, defaultdict

# ----------------------------------------------------------------------------------------------
# constants
# ----------------------------------------------------------------------------------------------
CAESAR_SHIFTS = (3, 4, 5, 7, 8, 9, 11, 13)   # encode shifts used by the injector (measured on train)
CAESAR_MINLEN = 4                             # shorter words are never shifted by the injector
CAESAR_RATE = 0.098                           # P(eligible non-keyword word is shifted) (measured on train)

# zero-width / invisible -> delete
_DELETE_CHARS = (
    [0x00AD, 0x034F, 0x061C, 0x115F, 0x17B4, 0x17B5, 0x180E, 0xFEFF, 0x20E3]
    + list(range(0x180B, 0x180E)) + list(range(0x200B, 0x2010)) + list(range(0x202A, 0x202F))
    + list(range(0x2060, 0x2070)) + list(range(0xFE00, 0xFE10)) + list(range(0xFFF0, 0xFFF9))
    + list(range(0xE0000, 0xE0080)) + list(range(0xE0100, 0xE01F0))
)
# blank-looking characters used as spacers -> space
_SPACE_CHARS = ([0x2800, 0x3164, 0x1160, 0xFFA0, 0x00A0, 0x2028, 0x2029, 0x202F, 0x205F, 0x3000, 0x0085]
                + list(range(0x2000, 0x200B)))

# homoglyphs (Cyrillic, Greek, IPA / small caps) -> Latin
_HOMO = {
    # Cyrillic lower
    'а': 'a', 'в': 'b', 'е': 'e', 'ё': 'e', 'і': 'i', 'ї': 'i', 'ј': 'j', 'к': 'k', 'м': 'm', 'н': 'h',
    'о': 'o', 'п': 'n', 'р': 'p', 'с': 'c', 'т': 't', 'у': 'y', 'х': 'x', 'ѕ': 's', 'ԁ': 'd', 'ԛ': 'q',
    'ԝ': 'w', 'һ': 'h', 'ӏ': 'l', 'ү': 'y', 'ь': 'b', 'г': 'r', 'и': 'u', 'л': 'n',
    # Cyrillic upper
    'А': 'A', 'В': 'B', 'Е': 'E', 'Ё': 'E', 'І': 'I', 'Ї': 'I', 'Ј': 'J', 'К': 'K', 'М': 'M', 'Н': 'H',
    'О': 'O', 'Р': 'P', 'С': 'C', 'Т': 'T', 'У': 'Y', 'Х': 'X', 'Ѕ': 'S', 'Ԁ': 'D', 'Ԛ': 'Q', 'Ԝ': 'W',
    'Һ': 'H', 'Ӏ': 'I', 'Ү': 'Y', 'П': 'N',
    # Greek
    'α': 'a', 'β': 'b', 'ε': 'e', 'ι': 'i', 'κ': 'k', 'ν': 'v', 'ο': 'o', 'ρ': 'p', 'τ': 't', 'υ': 'u',
    'χ': 'x', 'γ': 'y', 'ω': 'w', 'π': 'n', 'η': 'n', 'Α': 'A', 'Β': 'B', 'Ε': 'E', 'Ζ': 'Z', 'Η': 'H',
    'Ι': 'I', 'Κ': 'K', 'Μ': 'M', 'Ν': 'N', 'Ο': 'O', 'Ρ': 'P', 'Τ': 'T', 'Υ': 'Y', 'Χ': 'X', 'Λ': 'A',
    # Latin look-alikes / small capitals
    'ɑ': 'a', 'ı': 'i', 'ȷ': 'j', 'ℓ': 'l', 'ᴀ': 'a', 'ʙ': 'b', 'ᴄ': 'c', 'ᴅ': 'd', 'ᴇ': 'e', 'ꜰ': 'f',
    'ɢ': 'g', 'ʜ': 'h', 'ɪ': 'i', 'ᴊ': 'j', 'ᴋ': 'k', 'ʟ': 'l', 'ᴍ': 'm', 'ɴ': 'n', 'ᴏ': 'o', 'ᴘ': 'p',
    'ʀ': 'r', 'ꜱ': 's', 'ᴛ': 't', 'ᴜ': 'u', 'ᴠ': 'v', 'ᴡ': 'w', 'ʏ': 'y', 'ᴢ': 'z',
}
_TRANS = {c: None for c in _DELETE_CHARS}
_TRANS.update({c: ' ' for c in _SPACE_CHARS})
_TRANS.update({ord(k): v for k, v in _HOMO.items()})
_DELETE_SET = frozenset(chr(c) for c in _DELETE_CHARS)
_HOMO_SET = frozenset(_HOMO)

_COMBINING_AFTER_LATIN = re.compile(r'(?<=[A-Za-z])[\u0300-\u036f\u1ab0-\u1aff\u1dc0-\u1dff\u20d0-\u20ff\ufe20-\ufe2f]+')
_NON_ASCII = re.compile(r'[^\x00-\x7f]')

# bare domains: dot-separated labels (no '..'), and no bare '.org' ('goblok...org', 'y.org' = '... orang' in this data)
URL_RE = re.compile(r'(?:https?://|www\.)\S+|(?<![@\w.-])[\w-]+(?:\.[\w-]+)*\.(?:com|co\.id|id|net|tv|ly|me|io|go\.id|'
                    r'ac\.id)(?:/\S*)?\b', re.I)
MENTION_RE = re.compile(r'(?<![\w@])@[\w.]*\w')
_URL_PH, _MEN_PH = '\ue000', '\ue001'          # private-use placeholders (not \w -> never re-normalized)
LETTER = r'[^\W\d_]'
# dotted single letters: 'p.a.r.k.i.r. .l.i.a.r', 'B.P.J.S', 'l.i.m.b.a.h'
DOT_RUN_RE = re.compile(r'(?<![\w.])((?:%s\.(?: \.)?)+%s)(\.?)(?![\w])' % (LETTER, LETTER))
# spaced single letters (>= 3): 'b e g a l', 'J a l a n R u s a k'
SPACE_RUN_RE = re.compile(r'(?<![^\W_])((?:%s ){2,}%s)(?![^\W_])' % (LETTER, LETTER))
TOKEN_RE = re.compile(r'#[^\W_]\w*|[^\W_]+')     # hashtag or word
HASHTAG_RE = re.compile(r'#[^\W_]\w*')
WORD_RE = re.compile(r'[^\W_]+')
ELONG_RE = re.compile(r'(?<=[^\W\d_])([^\W\d_])\1{2,}')   # run of >=3 identical letters, not word-initial
WS_RE = re.compile(r'\s+')
REPEAT_SYM_RE = re.compile(r'([^\w\s])\1{3,}')          # '!!!!!' -> '!!!' ; 😭 x6 -> x3
# Injected trailing emoji: ~51% of rows (train AND test, comments AND captions) end with ' ' + one of these 11
# emojis, each ~4.6%, uniformly distributed and uncorrelated with every label (checked on train) -> pure noise.
INJECTED_EMOJI_RE = re.compile('\\s(?:\U0001F6D1|\u26A0\uFE0F?|\U0001F440|\u2757\uFE0F?|\U0001F644|\U0001F64F|'
                               '\U0001F621|\U0001F4A5|\U0001F6F5|\U0001F926\u200D\u2642\uFE0F?|\u203C\uFE0F?)\\s*$')

LEET = {'4': ('a',), '3': ('e',), '1': ('i', 'l'), '0': ('o',), '5': ('s',), '9': ('g',)}  # injector map (+1->l)
REDUP_RE = re.compile(r'([a-z][a-z0-9]*?[a-z])2([a-z]{0,6})')
ROMAN_RE = re.compile(r'(?=[mdclxvi]{2,}$)m{0,4}(?:cm|cd|d?c{0,3})(?:xc|xl|l?x{0,3})(?:ix|iv|v?i{0,3})$')  # VIII, XIII
# what may follow 'kata2' inside one token: clitics / suffixes / regional particles ('x' = plural marker 'kata2x')
REDUP_SUFFIXES = frozenset(
    ['', 'x', 'nya', 'ny', 'nyo', 'nye', 'na', 'ne', 'e', 'an', 'in', 'kan', 'kn', 'i', 'ku', 'mu', 'lah', 'kah', 'pun',
     'ken', 'keun', 'eun', 'aken', 'ke', 'no', 'ni', 'ko', 'mi', 'ka', 'annya', 'anya', 'ane', 'nyah', 'tah', 'nyalah'])
# Partial (prefixed) reduplication 'bertahun2' -> 'bertahun-tahun': hand-written list of frequent forms (base -> root)
REDUP_PARTIAL = {
    'seolah': 'olah', 'semena': 'mena', 'sewena': 'wena', 'sehari': 'hari', 'sekali': 'kali', 'sewaktu': 'waktu',
    'dimana': 'mana', 'kemana': 'mana', 'berbelit': 'belit', 'bertele': 'tele', 'berhati': 'hati', 'berlarut': 'larut',
    'berbondong': 'bondong', 'berduyun': 'duyun', 'berbeda': 'beda', 'berubah': 'ubah', 'berulang': 'ulang',
    'berputar': 'putar', 'berlomba': 'lomba', 'bersama': 'sama', 'berkata': 'kata', 'bertanya': 'tanya', 'berapi': 'api',
    'berkaca': 'kaca', 'berandai': 'andai', 'bermacam': 'macam', 'berlapis': 'lapis', 'terburu': 'buru',
    'terbirit': 'birit', 'terbahak': 'bahak', 'terbata': 'bata', 'terngiang': 'ngiang', 'tersengal': 'sengal',
    'terbungkuk': 'bungkuk', 'terbelit': 'belit', 'terheran': 'heran', 'tergesa': 'gesa', 'menghambur': 'hambur',
    'dihambur': 'hambur', 'mengada': 'ada', 'mengolok': 'olok', 'diolok': 'olok', 'menakut': 'nakut', 'menduga': 'duga',
    'memilih': 'milih', 'memilah': 'milah', 'mencoba': 'coba', 'menolak': 'nolak', 'membuang': 'buang',
    'menunggu': 'nunggu', 'menjawab': 'jawab', 'meninggi': 'tinggi', 'dikata': 'kata', 'dimarah': 'marah',
    'dibodoh': 'bodoh', 'dilempar': 'lempar', 'dipilih': 'pilih', 'dibagi': 'bagi', 'dicoba': 'coba',
    'ditanya': 'tanya', 'diganti': 'ganti', 'diurus': 'urus', 'dikirim': 'kirim', 'dibuang': 'buang', 'dielus': 'elus',
    'ditarik': 'tarik', 'dibalas': 'balas', 'ditunda': 'tunda', 'ditunjuk': 'tunjuk', 'disanjung': 'sanjung',
    'dirusak': 'rusak', 'dimaki': 'maki', 'dielu': 'elu', 'dicukup': 'cukup', 'dilimpah': 'limpah', 'dihalang': 'halang',
    'kedesa': 'desa', 'kekota': 'kota', 'kesana': 'sana',
}
# 'ber' + counting unit is always partial: berjam2 / beribu2 / berpuluh2 tahun
_BER_UNITS = ('tahun', 'taun', 'bulan', 'hari', 'jam', 'minggu', 'menit', 'detik', 'abad', 'kali', 'puluh', 'ratus',
              'ribu', 'juta', 'milyar', 'miliar', 'triliun', 'hektar', 'ton', 'kilo', 'meter', 'lembar', 'karung', 'truk',
              'hari', 'periode')
REDUP_UNIT = {p + u: u for u in _BER_UNITS for p in (('be',) if u[0] == 'r' else ('ber',))}
REDUP_UNIT.update({'ber' + u: u for u in _BER_UNITS})
# prefixed bases that are NOT partial reduplications even though base = prefix + known word
_REDUP_FULL_EXCEPT = frozenset(['kepala', 'keluar', 'kenapa', 'berdiri', 'terbaik', 'terlalu', 'tertentu', 'terjadi',
                                'terakhir', 'terutama', 'kepada', 'keadaan', 'kegiatan', 'kejadian', 'kerabat'])
# 'se' + adjective + '2nya' is the superlative 'sebaik-baiknya'; these 'se' words are not
# strings that are affixes / clitics more often than words: never split them off a camel-case hashtag piece
_AFFIX_LIKE = frozenset(['ter', 'ber', 'per', 'nya', 'kan', 'lah', 'kah', 'pun', 'men', 'mem', 'pen', 'pem', 'ken', 'nan',
                         'han', 'lan', 'ran', 'tan', 'wan', 'man'])
_REDUP_PREFIXES = frozenset(['ber', 'be', 'ter', 'se', 'di', 'ke', 'me', 'mem', 'men', 'meng', 'meny', 'pe', 'per'])
_HYPHEN_PAIR_RE = re.compile(r'(?<![a-z-])([a-z]{2,})-([a-z]{2,})(?![a-z-])')
_SE_NOT_SUPERLATIVE = frozenset(['sebelum', 'sesudah', 'setelah', 'semua', 'sendiri', 'sering', 'sekolah', 'selama',
                                 'sekitar', 'sesama', 'seluruh', 'sedang', 'sekarang', 'seperti', 'setiap', 'selalu',
                                 'sejak', 'semenjak', 'sebab', 'seberang', 'sepeda', 'senang', 'selesai', 'semangat',
                                 'sembuh', 'sebagai', 'sekali', 'setengah'])
CODE_RE = re.compile(r'[a-z]+\d{2,}|\d{2,}[a-z]+')
UNIT_RE = re.compile(
    r'^\d+(?:[.,]\d+)?(?:k|rb|ribu|jt|juta|m|t|an|x|kg|km|cm|mm|ml|gb|mb|kb|gr|g|hari|jam|th|thn|tahun|bln|bulan|'
    r'minggu|mgg|menit|mnt|detik|dtk|s|am|pm|wib|wita|wit|st|nd|rd|ribuan|jutaan|kali|orang|org|p|d|h|w|ha|'
    r'mw|kv|cc|lt|l|ton|unit|rt|rw|km2|m2|m3|persen|x24)$')
HARD_UNIT_RE = re.compile(r'^\d+(?:[.,]\d+)?(?:k|rb|ribu|jt|juta|m|t|x|an|jam|th|thn|bln|kg|km)$')  # never leet
ALNUM_KEEP = {'p3k', 'b3', 'k3', 'b2b', 'g20', 'g7', 'mp3', 'mp4', '3g', '4g', '5g', 's1', 's2', 's3', 'd1', 'd2',
              'd3', 'd4', 'k13', 'covid19', 'p2tp2a', 'pph21', 'r2', 'r4', 'tk1', 'tk2', 'x1', 'h2o', 'co2',
              'p5', '1x24', 'p4', 'p2', 'h1n1', 'h5n1', 'p3dn', 'b1', 'a1', 'e1', 'm2', 'm3', 'cm2', 'km2', 'mm2'}

# Small hand-written slang / abbreviation dictionary: only very frequent, (near) unambiguous forms in this corpus.
SLANG = {
    'yg': 'yang', 'yng': 'yang', 'krn': 'karena', 'karna': 'karena', 'krna': 'karena', 'utk': 'untuk',
    'untk': 'untuk', 'dgn': 'dengan', 'dg': 'dengan', 'dngn': 'dengan', 'gak': 'tidak', 'ga': 'tidak',
    'gk': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak', 'nggk': 'tidak', 'enggak': 'tidak', 'engga': 'tidak',
    'tdk': 'tidak', 'tak': 'tidak', 'sdh': 'sudah', 'udh': 'sudah', 'udah': 'sudah', 'blm': 'belum',
    'blum': 'belum', 'bgt': 'banget', 'bngt': 'banget', 'aja': 'saja', 'aj': 'saja', 'klu': 'kalau',
    'kalo': 'kalau', 'klo': 'kalau', 'jg': 'juga', 'jga': 'juga', 'sm': 'sama', 'org': 'orang', 'orng': 'orang',
    'tp': 'tapi', 'tpi': 'tapi', 'sy': 'saya', 'bs': 'bisa', 'bsa': 'bisa', 'lg': 'lagi', 'lgi': 'lagi',
    'lbh': 'lebih', 'msh': 'masih', 'skrg': 'sekarang', 'skrng': 'sekarang', 'trs': 'terus', 'pd': 'pada',
    'dlm': 'dalam', 'dll': 'dan lain-lain', 'dsb': 'dan sebagainya', 'tsb': 'tersebut', 'kmrn': 'kemarin',
    'gmn': 'gimana', 'knp': 'kenapa', 'hrs': 'harus', 'jd': 'jadi', 'jdi': 'jadi', 'pdhl': 'padahal',
    'bkn': 'bukan', 'sbg': 'sebagai', 'spt': 'seperti', 'kpd': 'kepada', 'thd': 'terhadap', 'tlg': 'tolong',
    'mrk': 'mereka', 'kyk': 'kayak', 'bnyk': 'banyak', 'byk': 'banyak', 'dpt': 'dapat', 'smua': 'semua',
    'gpp': 'tidak apa-apa', 'jln': 'jalan', 'jl': 'jalan', 't4': 'tempat', 'tmpt': 'tempat', 'sklh': 'sekolah',
    'msyrkt': 'masyarakat', 'pemkab': 'pemerintah kabupaten', 'pemkot': 'pemerintah kota',
    'pemprov': 'pemerintah provinsi', 'pemda': 'pemerintah daerah', 'se7': 'setuju', 'emg': 'emang',
    'bkin': 'bikin',
}

_EPS = 1e-12


def _is_nan(x):
    return x is None or (isinstance(x, float) and math.isnan(x))


# ----------------------------------------------------------------------------------------------
# stage 1-2 : character level
# ----------------------------------------------------------------------------------------------
def char_clean(text) -> str:
    """NFKC + invisible removal + spacer->space + homoglyphs + Latin diacritics stripping. NaN-safe."""
    if _is_nan(text):
        return ''
    s = str(text)
    if not s or not _NON_ASCII.search(s):
        return s
    s = unicodedata.normalize('NFKC', s)
    s = s.translate(_TRANS)
    if _NON_ASCII.search(s):
        s = unicodedata.normalize('NFD', s)
        s = _COMBINING_AFTER_LATIN.sub('', s)
        s = unicodedata.normalize('NFC', s)
        s = s.translate(_TRANS)
    return s


def _char_stats(text: str) -> Counter:
    """counts of character-level obfuscation in a raw text (for reporting only)."""
    c = Counter()
    if not _NON_ASCII.search(text):
        return c
    for ch in text:
        if ord(ch) < 128:
            continue
        if ch in _DELETE_SET:
            c['invisible_char'] += 1
        elif ch in _HOMO_SET:
            c['homoglyph_char'] += 1
        elif unicodedata.normalize('NFKC', ch) != ch and unicodedata.normalize('NFKC', ch).isascii():
            c['nfkc_char'] += 1   # fullwidth / math alphanumerics / superscripts
    return c


def shift_word(w: str, k: int) -> str:
    """Caesar-shift the lowercase ascii letters of w by k."""
    return ''.join(chr((ord(c) - 97 + k) % 26 + 97) if 'a' <= c <= 'z' else c for c in w)


def _apply_case(src: str, dst: str) -> str:
    """Transfer the capitalisation pattern of src onto dst (lower / UPPER / Title)."""
    letters = [c for c in src if c.isalpha()]
    if not letters or not dst:
        return dst
    if len(letters) > 1 and all(c.isupper() for c in letters):
        return dst.upper()
    if letters[0].isupper():
        return dst[:1].upper() + dst[1:]
    return dst


def _join_dotted(s: str):
    """'p.a.r.k.i.r. .l.i.a.r' -> 'parkir liar', 'B.P.J.S' -> 'BPJS'. Returns (text, n_fired)."""
    if '.' not in s:
        return s, 0
    n = [0]

    def rep(m):
        n[0] += 1
        return ' '.join(w.replace('.', '') for w in m.group(1).split('. .'))

    return DOT_RUN_RE.sub(rep, s), n[0]


def _elong_candidates(w: str, max_runs: int = 6, only_long: bool = False):
    """variants where each run (>=2) of identical letters becomes 1 or 2 letters (word-initial run untouched).
    only_long: only runs of >= 3 are shrunk (double letters of the original word are kept)."""
    parts = [m.group() for m in re.finditer(r'(.)\1*', w)]
    runs = [i for i, g in enumerate(parts) if len(g) >= (3 if only_long else 2) and i > 0]
    if len(runs) > max_runs:
        runs = [i for i in runs if len(parts[i]) >= 3][:max_runs]
    runs = set(runs)
    cands = ['']
    for i, g in enumerate(parts):
        opts = (g[0], g[0] * 2) if i in runs else (g,)
        cands = [c + o for c in cands for o in opts]
    return cands


# ----------------------------------------------------------------------------------------------
# char trigram LM (scores how word-like an a-z string is)
# ----------------------------------------------------------------------------------------------
class CharLM:
    """Interpolated char-trigram LM over a-z with BOS/EOS, trained on word TYPES."""

    def __init__(self, words_weights):
        V = 28  # 0 = BOS, 1..26 = a-z, 27 = EOS
        c3, c2, h2, h1, c1 = defaultdict(float), defaultdict(float), defaultdict(float), defaultdict(float), \
            defaultdict(float)
        tot = 0.0
        for w, wt in words_weights:
            ids = [0, 0] + [ord(c) - 96 for c in w] + [27]
            for i in range(2, len(ids)):
                a, b, c = ids[i - 2], ids[i - 1], ids[i]
                c3[(a, b, c)] += wt
                h2[(a, b)] += wt
                c2[(b, c)] += wt
                h1[b] += wt
                c1[c] += wt
                tot += wt
        tot = tot or 1.0
        l3, l2, l1, l0 = 0.55, 0.30, 0.13, 0.02
        self.table = [[[0.0] * V for _ in range(V)] for _ in range(V)]
        for a in range(V):
            for b in range(V):
                hab, hb = h2.get((a, b), 0.0), h1.get(b, 0.0)
                for c in range(1, V):
                    p1 = c1.get(c, 0.0) / tot
                    p2 = c2.get((b, c), 0.0) / hb if hb > 0 else p1
                    p3 = c3.get((a, b, c), 0.0) / hab if hab > 0 else p2
                    self.table[a][b][c] = math.log(l0 / 27 + l1 * p1 + l2 * p2 + l3 * p3)

    def logp(self, w: str) -> float:
        t = self.table
        a = b = 0
        s = 0.0
        for ch in w:
            c = ord(ch) - 96
            if c < 1 or c > 26:
                return -60.0
            s += t[a][b][c]
            a, b = b, c
        return s + t[a][b][27]


# ----------------------------------------------------------------------------------------------
# vocabulary
# ----------------------------------------------------------------------------------------------
class Vocab(dict):
    """dict word -> TRAIN frequency (after de-obfuscation, Caesar artefacts folded into their source word).
    Words from `extra_words` (pretrained tokenizer) absent from train are present with frequency 0.
    Metadata in .meta (shift statistics used by the Normalizer)."""

    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.meta = {}

    def __reduce__(self):  # keep .meta when pickled
        return (self.__class__, (dict(self),), {'meta': self.meta})


def load_default_extra_words(model_name: str = 'indobenchmark/indobert-base-p1') -> set:
    """Whole-word (alphabetic, no '##') tokens of a pretrained tokenizer vocab (allowed: pretrained weights+tokenizers)."""
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_name)
    return {w for w in tok.get_vocab() if re.fullmatch(r'[a-z]{2,}', w)}


def _pre_for_vocab(s: str) -> list:
    """deterministic steps only -> lowercase tokens (spaced runs / hashtags / urls / mentions dropped)."""
    s = char_clean(s)
    s = URL_RE.sub(' ', s)
    s = MENTION_RE.sub(' ', s)
    s = HASHTAG_RE.sub(' ', s)
    s, _ = _join_dotted(s)
    s = SPACE_RUN_RE.sub(' ', s)
    return [t.lower() for t in WORD_RE.findall(s)]


def build_vocab(train_texts, extra_words=None, min_freq: int = 2, verbose: bool = False) -> Vocab:
    """Word -> freq built from TRAIN text only, after the deterministic de-obfuscation steps.

    * only alphabetic ascii tokens are counted; elongated tokens are folded into their reduced form when that
      form is a known word;
    * Caesar artefacts are folded into their source: a type w is an artefact of v = shift(w,-k), k in
      CAESAR_SHIFTS, if v is at least as frequent and (a) w's count is compatible with Caesar noise from v and v
      is more word-like (char-LM gain), or (b) the char-LM gain is very large (>= 6), or (c) v is >= 20x more
      frequent and not less word-like. Rare types (count <= 3) whose best shift is extremely
      word-like (gain >= 8) but unseen are dropped (artefacts of words absent from train).
      The number of shifted occurrences of each source is kept in vocab.meta['shifted'] -> per-word shift rate
      (this is how the normalizer learns that topic keywords are never shifted by the injector);
    * extra_words (e.g. whole-word tokens of a pretrained tokenizer) are added with freq 0 if absent.
    """
    raw, elong = Counter(), Counter()
    cache = {}
    texts = [t for t in train_texts if not _is_nan(t)]
    for t in texts:
        toks = cache.get(t)
        if toks is None:
            toks = [w for w in _pre_for_vocab(t) if w.isalpha() and w.isascii()]
            cache[t] = toks
        for w in toks:
            if ELONG_RE.search(w):
                elong[w] += 1
            else:
                raw[w] += 1
    n_el = 0
    fold = {}
    for w, n in elong.items():
        best = max(_elong_candidates(w), key=lambda c: raw.get(c, 0))
        if raw.get(best, 0) >= max(2, n):
            raw[best] += n
            fold[w] = best
            n_el += 1
        elif n >= 5:
            raw[w] += n
    extra = set()
    if extra_words:
        extra = {str(w).lower() for w in extra_words}
        extra = {w for w in extra if w.isalpha() and w.isascii() and len(w) >= 2}
    lm0 = CharLM([(w, 1.0) for w, n in raw.items() if n >= 2 and len(w) >= 2] + [(w, 1.0) for w in extra])
    exp_per_shift = CAESAR_RATE / len(CAESAR_SHIFTS)
    lmc = {}

    def lmp(x):
        r = lmc.get(x)
        if r is None:
            r = lmc[x] = lm0.logp(x)
        return r

    artefacts, dropped = {}, set()
    for w, n in raw.items():
        if len(w) < CAESAR_MINLEN:
            continue
        lw = lmp(w)
        best, unseen_gain = None, -1e9
        for k in CAESAR_SHIFTS:
            v = shift_word(w, -k)
            cv = raw.get(v, 0)
            gain = lmp(v) - lw
            if cv == 0:
                unseen_gain = max(unseen_gain, gain)
                continue
            if cv < max(2, n):
                continue
            e = cv * exp_per_shift   # expected number of occurrences of w produced by shifting v
            ok = ((gain >= 0.5 and n <= 3.0 * e + 2) or (gain >= 3.0 and n <= 6.0 * e + 3) or gain >= 6.0
                  or (cv >= 20 * n and gain >= 0.0))
            if ok and (best is None or cv > best[1]):
                best = (v, cv)
        if best is not None:
            artefacts[w] = best[0]
        elif n <= 3 and unseen_gain >= 8.0 and w not in extra:
            dropped.add(w)
    shifted = Counter()
    for w, v in artefacts.items():
        seen = {w}
        while v in artefacts and v not in seen:  # resolve chains
            seen.add(v)
            v = artefacts[v]
        n = raw.pop(w)
        raw[v] += n
        shifted[v] += n
    for w in dropped:
        raw.pop(w, None)
        fold[w] = None
    vocab = Vocab({w: n for w, n in raw.items() if n >= min_freq})
    # word bigrams (after folding) -> used by the segmentation DP (e.g. spaced 'r u m a h s a k i t')
    for w, v in artefacts.items():
        seen = {w}
        while v in artefacts and v not in seen:
            seen.add(v)
            v = artefacts[v]
        fold[w] = v
    big = Counter()
    for t in texts:
        prev = None
        for w in cache[t]:
            w = fold.get(w, w)
            if w is None or w not in vocab:
                prev = None
                continue
            if prev is not None:
                big[(prev, w)] += 1
            prev = w
    bigrams = {k: n for k, n in big.items() if n >= 2}
    # hyphenated reduplications written in full in TRAIN text: partial 'bertahun-tahun' vs full 'dimana-dimana'
    part, full = Counter(), Counter()
    for t in set(texts):
        s = char_clean(t)
        if '-' not in s:
            continue
        for a, b in _HYPHEN_PAIR_RE.findall(s.lower()):
            if a == b:
                full[a] += 1
            elif a.endswith(b) and 2 <= len(a) - len(b) <= 4 and a[:len(a) - len(b)] in _REDUP_PREFIXES:
                part[(a, b)] += 1
    redup_partial = {a: b for (a, b), n in part.items() if n > full.get(a, 0)}
    n_extra = 0
    for w in extra:
        if w not in vocab:
            vocab[w] = 0
            n_extra += 1
    vocab.meta = {'shifted': {v: shifted[v] for v in vocab if shifted.get(v)}, 'artefacts': artefacts,
                  'bigrams': bigrams, 'redup_partial': redup_partial,
                  'dropped': dropped, 'n_elong_folded': n_el, 'n_extra': n_extra, 'min_freq': min_freq}
    if verbose:
        print(f'[build_vocab] types={len(raw)} vocab={len(vocab)} (train>={min_freq}: {len(vocab) - n_extra}, '
              f'extra-only={n_extra}) caesar artefacts folded={len(artefacts)} dropped={len(dropped)} '
              f'elongations folded={n_el}')
    return vocab


# ----------------------------------------------------------------------------------------------
# Normalizer
# ----------------------------------------------------------------------------------------------
class Normalizer:
    """Callable text normalizer.

    Parameters
    ----------
    vocab : dict word->freq (preferably the Vocab returned by build_vocab)
    lowercase : True -> lowercase output (default; most Indonesian BERTs are uncased). False keeps case
                (decoded words inherit the capitalisation pattern of the obfuscated token).
    slang : dict to override the built-in SLANG dictionary ({} disables expansion).
    oov_mass : prior mass of real-but-unknown words (scales the char-LM probability of OOV strings).
    caesar_margin : nats the best decoding must beat the identity hypothesis by.
    lm_guard : a decoding to a known word is rejected if it is this much LESS word-like (char-LM) than the token.
    oov_decode_margin / oov_decode_minlen : decoding to a word unseen in the vocab (names...) requires a large
                char-LM margin and a minimum length.
    seg_* : penalties of the segmentation DP (OOV piece / pretrained-only piece / per split).
    strip_injected_emoji : drop the single injected trailing emoji (' 🛑', ' ⚠️', ' 👀', ' ❗', ' 🙄', ' 🙏', ' 😡',
                ' 💥', ' 🛵', ' 🤦‍♂️', ' ‼️'): label-independent noise added to ~51% of rows. Other emojis are kept.
    """

    def __init__(self, vocab: dict, lowercase: bool = True, slang: dict | None = None, oov_mass: float = 0.04,
                 extra_count_id: float = 1.0, extra_count_tgt: float = 0.3, caesar_margin: float = 1.0,
                 lm_guard: float = -1.0, oov_decode_margin: float = 4.0, oov_decode_minlen: int = 6,
                 seg_oov_pen: float = 3.0, seg_extra_pen: float = 0.0, seg_split_pen: float = 2.5,
                 strip_injected_emoji: bool = True, cache_size: int = 3_000_000):
        self.vocab = vocab
        self.lowercase = lowercase
        self.strip_injected_emoji = strip_injected_emoji
        self.slang = dict(SLANG if slang is None else slang)
        meta = getattr(vocab, 'meta', {}) or {}
        self.shifted = meta.get('shifted', {})
        self.bigrams = meta.get('bigrams', {})
        self.redup_partial = dict(meta.get('redup_partial', {}))   # learned from train hyphenated forms
        self.redup_partial.update(REDUP_PARTIAL)
        self.redup_partial.update(REDUP_UNIT)
        self.N = float(sum(v for v in vocab.values() if v > 0)) or 1.0
        self.log_oov = math.log(oov_mass)
        self.extra_count_id = extra_count_id
        self.extra_count_tgt = extra_count_tgt
        self.caesar_margin = caesar_margin
        self.lm_guard = lm_guard
        self.oov_decode_margin = oov_decode_margin
        self.oov_decode_minlen = oov_decode_minlen
        self.seg_oov_pen, self.seg_extra_pen, self.seg_split_pen = seg_oov_pen, seg_extra_pen, seg_split_pen
        self.lm = CharLM([(w, 1.0) for w in vocab if len(w) >= 2 and w.isalpha() and w.isascii()])
        self.log_keep = math.log(1.0 - CAESAR_RATE)
        self.log_shift_oov = math.log(CAESAR_RATE / len(CAESAR_SHIFTS))
        self._rho_a = 20.0
        self.cache_size = cache_size
        self._tok_cache, self._txt_cache, self._seg_cache, self._caesar_cache, self._lm_cache = {}, {}, {}, {}, {}

    @property
    def keep_case(self):
        return not self.lowercase

    # -------------------------------------------------------------- scoring helpers
    def _lm(self, w):
        r = self._lm_cache.get(w)
        if r is None:
            r = self._lm_cache[w] = self.lm.logp(w)
        return r

    def _logp_id(self, w):
        """log prior of w being a real word (identity hypothesis)."""
        n = self.vocab.get(w)
        if n is None:
            return self.log_oov + self._lm(w)
        return math.log((n if n > 0 else self.extra_count_id) / self.N)

    def _logp_tgt(self, v):
        """log prior that a shifted token decodes to v: P(v) * P(v gets shifted by a given k)."""
        n = self.vocab.get(v)
        if n is None:
            return None
        if n <= 0:
            return math.log(self.extra_count_tgt / self.N) + self.log_shift_oov
        s = self.shifted.get(v, 0)
        rho = (s + self._rho_a * CAESAR_RATE) / (n + self._rho_a)   # Bayesian per-word shift rate
        return math.log(n / self.N) + math.log(rho / len(CAESAR_SHIFTS) + _EPS)

    def caesar_scored(self, w: str):
        """w: lowercase ascii alpha. Returns (decoded, score_decoded - score_identity) or (None, 0)."""
        r = self._caesar_cache.get(w)
        if r is not None:
            return r
        r = (None, 0.0)
        n = self.vocab.get(w)
        if len(w) >= CAESAR_MINLEN and (n is None or n < 50):
            s_id = self._logp_id(w) + self.log_keep
            lw = self._lm(w)
            best, best_s = None, -1e18
            for k in CAESAR_SHIFTS:
                v = shift_word(w, -k)
                lp = self._logp_tgt(v)
                if lp is None or self._lm(v) - lw < self.lm_guard:
                    continue
                if lp > best_s:
                    best, best_s = v, lp
            if best is not None and best_s - s_id > self.caesar_margin:
                r = (best, best_s - s_id)
            elif n is None and len(w) >= self.oov_decode_minlen:
                # decoding to a string unseen in the vocab (e.g. a name absent from train): char-LM evidence only
                bv, bl = None, -1e18
                for k in CAESAR_SHIFTS:
                    v = shift_word(w, -k)
                    lv = self._lm(v)
                    if lv > bl:
                        bv, bl = v, lv
                gain = bl + self.log_shift_oov - (lw + self.log_keep)
                if gain > self.oov_decode_margin:
                    r = (bv, gain)
        self._caesar_cache[w] = r
        return r

    def caesar_decode(self, w: str):
        return self.caesar_scored(w)[0]

    def leet_candidates(self, w: str, limit: int = 64):
        cands = ['']
        for ch in w:
            opts = LEET.get(ch, (ch,))
            cands = [c + o for c in cands for o in opts]
            if len(cands) > limit:
                cands = cands[:limit]
        return cands

    # -------------------------------------------------------------- token normalisation
    def norm_token(self, tok: str):
        """tok: a [^\\W_]+ token (any case). Returns (normalised cased string, kinds tuple)."""
        r = self._tok_cache.get(tok)
        if r is None:
            r = self._norm_token(tok)
            if len(self._tok_cache) < self.cache_size:
                self._tok_cache[tok] = r
        return r

    def _norm_token(self, tok: str):
        lw = tok.lower()
        if not lw.isascii() or lw.isdigit():
            return tok, ()
        if any(c.isdigit() for c in lw):
            r = self._norm_alnum(lw)
            if r is None:
                return tok, ()
            out, kinds = r
        else:
            out, kinds = self._norm_alpha(lw)
        if out == lw:
            return tok, ()
        return _apply_case(tok, out), kinds

    def _final(self, w: str, kinds: list):
        s = self.slang.get(w)
        if s is not None:
            kinds.append('slang')
            return s, tuple(kinds)
        return w, tuple(kinds)

    def _norm_alpha(self, lw: str, allow_caesar: bool = True):
        kinds = []
        if lw in self.slang:
            return self._final(lw, kinds)
        n = self.vocab.get(lw)
        if ROMAN_RE.match(lw):              # 'kelas VIII', 'XIII' are not elongations / Caesar artefacts
            return self._final(lw, kinds)
        if ELONG_RE.search(lw) and (n is None or n < 5):
            kinds.append('elong')
            best, best_s, via_caesar = None, -1e18, False
            minimal = _elong_candidates(lw, only_long=True)       # 'masukkannnn' -> masukkan / masukkann
            full = _elong_candidates(lw)
            hit = [c for c in minimal if c in self.vocab or c in self.slang]
            if hit:   # keep the original double letters unless a full reduction is a far more frequent word
                fmax = max(self.vocab.get(c, 0) for c in full)
                hmax = max(self.vocab.get(c, 0) if c not in self.slang else 1000 for c in hit)
                if fmax >= 10 * max(hmax, 1):
                    hit = []
            for c in (hit or full):
                if c in self.slang:
                    sc = math.log(50.0 / self.N)
                    cand, cz = c, False
                elif c in self.vocab:
                    sc, cand, cz = self._logp_id(c), c, False
                elif allow_caesar:
                    d, gain = self.caesar_scored(c)
                    if d is None:
                        continue
                    sc, cand, cz = self._logp_tgt(d) if d in self.vocab else self._logp_id(c) + gain, d, True
                else:
                    continue
                if sc > best_s:
                    best, best_s, via_caesar = cand, sc, cz
            if best is None:
                best = ELONG_RE.sub(lambda m: m.group(1), lw)
            if via_caesar:
                kinds.append('caesar')
            return self._final(best, kinds)
        if allow_caesar and len(lw) >= CAESAR_MINLEN and (n is None or n < 50):
            d = self.caesar_decode(lw)
            if d is not None:
                kinds.append('caesar')
                return self._final(d, kinds)
        return self._final(lw, kinds)

    def _leet_caesar(self, cands):
        """best Caesar decoding over leet candidates -> (decoded, gain, vocab_count or -1) or None.
        Candidates are compared on the ABSOLUTE score of the decoded word (a gain is relative to its own source string,
        which would favour the least word-like '1'->'l' variant); train words first, then '1'->'i' readings.
        Elongated candidates are skipped (handled by the elongation path)."""
        best, key = None, None
        for c in cands:
            if len(c) < CAESAR_MINLEN or not c.isalpha() or ELONG_RE.search(c):
                continue
            d, g = self.caesar_scored(c)
            if d is None:
                continue
            n = self.vocab.get(d)
            s = self._logp_tgt(d) if n is not None else self.log_oov + self._lm(d)
            k = (n is not None and n > 0, s, c.count('i'))
            if key is None or k > key:
                best, key = (d, g, n if n is not None else -1), k
        return best

    def _redup(self, base: str, suffix: str):
        """'kata' + 'nya' -> 'kata-katanya'; prefixed bases are reduplicated on the root ('bertahun' -> 'bertahun-tahun',
        'sebaik'+'nya' -> 'sebaik-baiknya'); 'ke2'/'ber2' -> 'kedua'/'berdua'; a glued known word is split off
        ('toko2itu' -> 'toko-toko itu'); the plural marker 'x' is dropped ('gara2x' -> 'gara-gara')."""
        if base in ('ke', 'ber') and suffix in ('', 'nya'):
            return base + 'dua' + suffix
        tail = ''
        if suffix not in REDUP_SUFFIXES:   # a solid word glued after the '2' (not a suffix typo like 'mudah2n/2han')
            sn = self.slang.get(suffix)
            if sn is None and (suffix == 'di' or (len(suffix) >= 3 and self.vocab.get(suffix, 0) >= 20
                                                  and not (len(suffix) == 3 and suffix.endswith('an')))):
                sn = suffix
            if sn is not None:
                suffix, tail = '', ' ' + sn
        if suffix == 'x':
            suffix = ''
        root = self.redup_partial.get(base)
        if root is None and base not in _REDUP_FULL_EXCEPT:
            V = self.vocab
            if base.startswith('se') and suffix in ('nya', 'ny') and base not in _SE_NOT_SUPERLATIVE \
                    and len(base) >= 5 and V.get(base[2:], 0) >= 5:
                root = base[2:]                               # sebaik-baiknya, setinggi-tingginya
            else:
                for p, minlen in (('ber', 4), ('di', 4), ('ter', 5), ('ke', 5)):   # not terbang->bang, kenang->nang
                    r = base[len(p):]
                    if base.startswith(p) and len(r) >= minlen and V.get(r, 0) >= 20 and V.get(base, 0) < V[r]:
                        root = r                              # berjalan-jalan, dipilih-pilih, kedesa-desa
                        break
        return f'{base}-{root or base}{suffix}{tail}'

    def _norm_alnum(self, lw: str, _depth: int = 0):
        """letters+digits token -> (new lowercase string, kinds) or None (keep as is)."""
        if lw in ALNUM_KEEP or HARD_UNIT_RE.match(lw) or lw.startswith('rp'):
            return None
        if lw in self.slang:
            return self._final(lw, [])
        if UNIT_RE.match(lw):   # soft unit ('19d','50514l','094h'): only a confident leet decode overrides it
            if all((c.isalpha() or c in LEET) for c in lw):
                best = max(self.leet_candidates(lw), key=lambda c: self.vocab.get(c, -1))
                if len(best) >= 3 and self.vocab.get(best, -1) >= 5:
                    return self._final(best, ['leet'])
            return None
        if self.vocab.get(lw, 0) >= 5:
            return None
        leetable = all((c.isalpha() or c in LEET) for c in lw)
        cands = None
        if leetable:
            cands = self.leet_candidates(lw)
            best = max(cands, key=lambda c: (self.vocab.get(c, -1), c.count('i')))
            if len(best) >= 3 and self.vocab.get(best, -1) >= 2:
                return self._final(best, ['leet'])
            # leet on top of Caesar with strong evidence (frequent train word, large margin) even if the token looks
            # like a code ('umzm51' -> mereka, 'xcv91' -> punya, 'v0015' -> nggak)
            if len(lw) >= CAESAR_MINLEN and sum(c.isalpha() for c in lw) >= 2:
                lc = self._leet_caesar(cands)
                if lc is not None and lc[2] >= 20 and lc[1] >= 4.0:
                    return self._final(lc[0], ['leet', 'caesar'])
        if CODE_RE.fullmatch(lw):          # usernames / flight numbers / counts
            return None
        m = REDUP_RE.fullmatch(lw) if _depth == 0 else None   # reduplication: kata2 / kata2nya / mudah2an
        if m:
            base = m.group(1)
            if any(c.isdigit() for c in base):
                r = self._norm_alnum(base, _depth + 1)        # depth-limited (no RecursionError on 'a2a2a2...')
                base = r[0] if r else base
            if base.isalpha():
                base = self._norm_alpha(base, allow_caesar=False)[0]
            if ' ' not in base and not any(ch.isdigit() for ch in base):   # undecodable code: no reduplication
                return self._redup(base, m.group(2)), ('redup',)
        if not leetable:
            return None
        # injector leet maps exactly a/e/i/o/s/g -> 4/3/1/0/5/9: a true leet token contains none of these letters
        if len(lw) >= 3 and any(c.isalpha() for c in lw) and not any(c in 'aeiosg' for c in lw):
            lc = self._leet_caesar(cands) if len(lw) >= CAESAR_MINLEN else None   # leet on top of a Caesar shift
            if lc is not None and lc[2] >= 0:                      # ... decoding into a known word
                return self._final(lc[0], ['leet', 'caesar'])
            known = [c for c in cands if c in self.vocab]          # e.g. pretrained-vocab word
            if known:
                return self._final(max(known, key=lambda c: self.vocab[c]), ['leet'])
            out, kinds = self._norm_alpha(cands[0], allow_caesar=False)   # elongation / slang ('d1r11111' -> diri)
            if lc is not None and not (out in self.vocab or 'slang' in kinds):
                # decoding into a word unseen in train, as the alpha path does ('03n3hm3r' -> kajadian)
                return self._final(lc[0], ['leet', 'caesar'])
            return out, ('leet',) + kinds
        return None

    # -------------------------------------------------------------- segmentation (vocab DP)
    def _seg_logp(self, prev, w: str) -> float:
        """log P(w | prev) for segmentation: interpolated bigram over the train vocab; OOV pieces via char LM."""
        n = self.vocab.get(w)
        if n is None:
            return self.log_oov + self._lm(w) - self.seg_oov_pen
        lp = math.log(n / self.N) if n > 0 else math.log(self.extra_count_tgt / self.N) - self.seg_extra_pen
        if len(w) <= 2:
            lp -= 4.0 if len(w) == 2 else 6.0
        if prev is not None:
            c = self.bigrams.get((prev, w))
            if c:
                return math.log(0.5 * c / max(self.vocab.get(prev, 1), 1) + 0.5 * math.exp(lp))
            return lp + math.log(0.5)
        return lp

    def segment(self, s: str, bonus_pos=frozenset(), max_len: int = 24, split_pen: float | None = None):
        """Word segmentation of lowercase alpha string s (approximate Viterbi, bigram-aware, OOV via char LM).
        bonus_pos: indices where a split is encouraged (camel-case boundaries)."""
        key = (s, bonus_pos)
        r = self._seg_cache.get(key)
        if r is not None:
            return r
        L = len(s)
        if split_pen is None:
            split_pen = self.seg_split_pen
        best = [0.0] + [1e18] * L
        back = [0] * (L + 1)
        last = [None] * (L + 1)
        for j in range(1, L + 1):
            for i in range(max(0, j - max_len), j):
                if best[i] >= 1e17:
                    continue
                w = s[i:j]
                c = best[i] - self._seg_logp(last[i], w)
                if i > 0:
                    c += split_pen - (8.0 if i in bonus_pos else 0.0)
                if c < best[j]:
                    best[j], back[j], last[j] = c, i, w
        out, j = [], L
        while j > 0:
            out.append(s[back[j]:j])
            j = back[j]
        out.reverse()
        self._seg_cache[key] = out
        return out

    def _join_spaced(self, s: str, kinds: Counter) -> str:
        def rep(m):
            joined = m.group(1).replace(' ', '')
            low = joined.lower()
            kinds['spaced_join'] += 1
            bonus = frozenset()
            if not (joined.isupper() or joined.islower()):   # camel boundaries in a mixed-case run
                bonus = frozenset(i for i in range(1, len(joined)) if joined[i].isupper() and joined[i - 1].islower())
            words = self.segment(low, bonus)
            if len(words) > 1:
                kinds['spaced_split'] += 1
            out, p = [], 0
            for w in words:
                out.append(joined[p:p + len(w)])
                p += len(w)
            return ' '.join(out)

        return SPACE_RUN_RE.sub(rep, s)

    def _known(self, w):
        return self.vocab.get(w, 0) > 0 or w in self.slang

    def _piece_ok(self, w):
        n = self.vocab.get(w)
        if n is None or len(w) < 2:
            return False
        if len(w) == 2:
            return n >= 50
        return n >= 2 or (n == 0 and len(w) >= 5)

    def _hashtag(self, tag: str, kinds: Counter) -> str:
        kinds['hashtag'] += 1
        if '_' in tag:
            return ' '.join(self._hashtag_part(t, kinds) for t in tag.split('_') if t)
        return self._hashtag_part(tag, kinds)

    def _camel_split(self, tag: str):
        """'SaveBPJSKesehatan' -> Save|BPJS|Kesehatan ; 'BPJSkesehatan' -> BPJS|kesehatan ;
        'TOTOPeduliBumi' -> TOTO|Peduli|Bumi (an upper-case run followed by lower case is resolved with the vocab)."""
        out = []
        for p in re.findall(r'[A-Z]+[a-z]*|[a-z]+|\d+|[^\W\d_]+', tag):
            m = re.fullmatch(r'([A-Z]{2,})([a-z]+)', p)
            if m:
                up, low = m.groups()
                a = (up[:-1], up[-1] + low)       # TOTO + Peduli
                b = (up, low)                      # BPJS + kesehatan ; ATCS + parodi ; SAR + untuksemua
                sc = self._camel_score
                ka = (min(sc(a[0]), sc(a[1])), sc(a[1]))
                kb = (min(sc(b[0]), sc(b[1])), sc(b[1]))
                out.extend(a if ka >= kb else b)
            else:
                out.append(p)
        return out

    def _camel_score(self, piece: str):
        """vocab count of a camel piece; an unknown lowercase concatenation of solid words ('untuksemua') scores as
        its weakest word; else -1."""
        lo = piece.lower()
        n = self.vocab.get(lo)
        if n is not None:
            return n
        if lo.isalpha() and len(lo) >= 6:
            ws = self._split_compound(lo)
            if len(ws) > 1:
                return min(self.vocab.get(w, 0) for w in ws)
        return -1

    def _split_compound(self, lo: str):
        """vocab-DP split of a concatenation ('jalanrusak' -> ['jalan','rusak']); rare known compounds
        ('rumahsakit', count < 20) are split too when every piece is a solid word."""
        n = self.vocab.get(lo)
        if not lo.isalpha() or len(lo) < 4 or lo in self.slang or (n is not None and (n == 0 or n >= 20)):
            return [lo]   # pretrained-vocab words (n == 0, e.g. 'kanada') and frequent words stay whole
        words = self.segment(lo)
        if len(words) > 1 and all(self._piece_ok(w) for w in words):
            return words
        return [lo]

    def _hashtag_part(self, tag: str, kinds: Counter) -> str:
        camel = re.search(r'[a-z][A-Z]|[A-Z]{2,}[a-z]', tag) is not None
        parts = self._camel_split(tag) if camel else (re.findall(r'\d+|[^\W\d_]+', tag) or [tag])
        if not camel and len(parts) > 1 and WORD_RE.fullmatch(tag):
            out, k = self.norm_token(tag)        # whole-tag leet ('#k3m3n505')
            if out.lower() != tag.lower():
                parts = [tag]
        res = []
        for p in parts:
            lp = p.lower()
            if not lp.isascii() or not lp.isalpha() and not WORD_RE.fullmatch(p):
                res.append(p)
                continue
            out, k = self.norm_token(p)
            kinds.update(k)
            lo = out.lower()
            if camel:                              # camel pieces are user-delimited words ...
                lo = out.lower()
                if not p.isupper() and len(lo) >= 6 and lo.isalpha() and lo not in self.vocab:
                    words = self._split_compound(lo)   # ... unless a piece glues frequent words: 'Tuntasdan'
                    if len(words) > 1 and all(len(w) >= 3 and self.vocab.get(w, 0) >= 5 and w not in _AFFIX_LIKE
                                              for w in words):
                        kinds['hashtag_split'] += 1
                        res.extend(words)
                        continue
                res.append(out)
                continue
            words = self._split_compound(lo)
            if len(words) > 1:
                kinds['hashtag_split'] += 1
                res.extend(words)
            else:
                res.append(out)
        return ' '.join(res)

    # -------------------------------------------------------------- text level
    def __call__(self, text) -> str:
        if _is_nan(text):
            return ''
        r = self._txt_cache.get(text)
        if r is None:
            r = self._normalize(text)
            if len(self._txt_cache) < self.cache_size:
                self._txt_cache[text] = r
        return r[0]

    def transform_counts(self, texts) -> Counter:
        """Occurrence counts of every transformation over `texts` (with multiplicity)."""
        tot = Counter()
        for t in texts:
            if _is_nan(t):
                continue
            self(t)
            tot.update(self._txt_cache[t][1])
        return tot

    def _normalize(self, text):
        text = str(text)
        kinds = _char_stats(text)
        if self.strip_injected_emoji:
            text, n = INJECTED_EMOJI_RE.subn('', text)
            kinds['injected_emoji'] += n
        s = char_clean(text)
        s, n = URL_RE.subn(f' {_URL_PH} ', s)
        kinds['url'] += n
        s, n = MENTION_RE.subn(f' {_MEN_PH} ', s)
        kinds['mention'] += n
        s, n = _join_dotted(s)
        kinds['dotted_join'] += n
        s = self._join_spaced(s, kinds)

        def tok(m):
            g = m.group()
            if g[0] == '#':
                return ' ' + self._hashtag(g[1:], kinds) + ' '
            out, k = self.norm_token(g)
            if k:
                kinds.update(k)
            return out

        s = TOKEN_RE.sub(tok, s)
        s = REPEAT_SYM_RE.sub(lambda m: m.group(1) * 3, s)
        s = WS_RE.sub(' ', s).strip()
        if self.lowercase:
            s = s.lower()
        s = s.replace(_URL_PH, 'url').replace(_MEN_PH, '@user')
        return s, kinds


def normalize_series(series, normalizer):
    """Apply `normalizer` to a pandas Series mapping unique values only (NaN -> '')."""
    import pandas as pd
    s = pd.Series(series)
    mp = {u: normalizer(u) for u in s.dropna().unique()}
    return s.map(mp).fillna('').astype(str)


def default_normalizer(train_csv: str = '../penyisihan/data/train.csv', lowercase: bool = True,
                       verbose: bool = False):
    """Convenience: build the production vocab from TRAIN (Teks + unique captions) + indobert-base-p1 whole words
    and return (vocab, Normalizer). ~8 s."""
    import pandas as pd
    tr = pd.read_csv(train_csv)
    texts = tr.Teks.tolist() + tr.Judul_Caption.dropna().drop_duplicates().tolist()
    vocab = build_vocab(texts, load_default_extra_words(), verbose=verbose)
    return vocab, Normalizer(vocab, lowercase=lowercase)

Overwriting textnorm.py


## 2. Data, label, dan utilitas
Label 4 kelas diubah menjadi: `REL` (sel relevan), `YS` (multi-hot sentimen yang muncul pada komentar), dan `OK` (komentar yang punya minimal satu pilar relevan — hanya baris ini yang dipakai melatih model sentimen). Fold = Multilabel-Stratified Group K-Fold per postingan dari babak penyisihan (`artifacts_final/penyisihan/folds.csv`).

In [3]:
def lg(x):
    x = np.clip(x, EPS, 1 - EPS); return np.log(x / (1 - x))


def load_data(normalize=True):
    """return dict: train, test, sample, B (N,9 string), REL (N,9 bool), YS (N,3 multi-hot sentimen), OK (baris ber-sentimen),
    IS_X, FOLD, GROUP"""
    train = pd.read_csv(f"{DATA_DIR}/train.csv"); test = pd.read_csv(f"{DATA_DIR}/test.csv"); sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
    assert list(sample.columns) == ["ID"] + LABELS and (sample.ID.values == test.ID.values).all()
    B = train[LABELS].values.astype(str); assert set(np.unique(B)) <= set(C4), np.unique(B)
    REL = B != "TidakRelevan"
    YS = np.stack([((B == s) & REL).any(1) for s in SENT], 1).astype(int)          # multi-hot: sentimen yang muncul di baris itu
    folds = pd.read_csv(f"{V9}/folds.csv"); assert (folds.ID.values == train.ID.values).all()
    d = dict(train=train, test=test, sample=sample, B=B, REL=REL, YS=YS, OK=REL.any(1), IS_X=train.Judul_Caption.isna().values,
             FOLD=folds.fold.values, GROUP=folds.group.astype(str).values)
    if normalize:
        cp = f"{ART}/norm_cache.joblib"
        if os.path.exists(cp):
            nc = joblib.load(cp)
        else:
            import textnorm as T
            t0 = time.time(); vocab = joblib.load(f"{V9}/models/textnorm_vocab.joblib"); norm = T.Normalizer(vocab, lowercase=False)
            nc = {}
            for nm, df in (("train", train), ("test", test)):
                nc[nm + "_teks_c"] = list(T.normalize_series(df.Teks, norm)); nc[nm + "_cap_c"] = list(T.normalize_series(df.Judul_Caption, norm))
            joblib.dump(nc, cp, compress=3); print(f"normalisasi train+test selesai dalam {time.time() - t0:.0f}s")
        for nm, df in (("train", train), ("test", test)):
            df["teks_c"], df["cap_c"] = nc[nm + "_teks_c"], nc[nm + "_cap_c"]
            df["teks_l"], df["cap_l"] = df.teks_c.str.lower(), df.cap_c.str.lower()
    return d


# ---------------------------------------------------------------- tahap 1: relevansi dari v9
def post_shrink(Q, l1_list, groups, w=0.2, iters=60):                # identik dengan notebook v9 (sel 9.1)
    gid = pd.factorize(pd.Series(groups).astype(str))[0]; G = gid.max() + 1; cnt = np.bincount(gid, minlength=G)[:, None]
    gmean = lambda X: np.stack([np.bincount(gid, weights=X[:, j], minlength=G) for j in range(X.shape[1])], 1) / cnt
    tgt = np.clip((1 - w) * gmean(Q) + w * np.mean([gmean(P) for P in l1_list], 0), EPS, 1 - EPS)
    Z, lo, hi = lg(Q), np.full(tgt.shape, -15.0), np.full(tgt.shape, 15.0)
    for _ in range(iters):
        mid = (lo + hi) / 2; up = gmean(1 / (1 + np.exp(-(Z + mid[gid])))) < tgt
        lo, hi = np.where(up, mid, lo), np.where(up, hi, mid)
    return 1 / (1 + np.exp(-(Z + ((lo + hi) / 2)[gid])))


def relevance_v9(d):
    """(Q_oof [N_train,9] dengan NaN pada baris X, Q_test [N_test,9], threshold) - prob relevansi final v9 (stacker + post-shrink)."""
    th = json.load(open(f"{V9}/l2/thresholds.json")); keys = th.get("post_shrink_l1", []); w = th.get("post_shrink_w", 0.0)
    nx = ~d["IS_X"]; Qo = np.load(f"{V9}/l2/l2_oof.npy"); Qt = np.load(f"{V9}/l2/l2_test.npy")
    Q_oof = np.full(Qo.shape, np.nan)
    if w > 0:
        Q_oof[nx] = post_shrink(Qo[nx], [np.load(f"{V9}/l1/{k}_oof.npy")[nx] for k in keys], d["GROUP"][nx], w)
        Q_test = post_shrink(Qt, [np.load(f"{V9}/l1/{k}_test.npy") for k in keys], d["test"].Judul_Caption.astype(str).values, w)
    else:
        Q_oof[nx] = Qo[nx]; Q_test = Qt
    return Q_oof, Q_test, float(np.mean(th["thresholds"]))


# ---------------------------------------------------------------- metrik resmi & perakitan
def score4(Ytrue, Ypred, detail=False):
    """Mean Column-wise Macro-F1 atas 4 kelas (metrik leaderboard final)."""
    per = np.array([f1_score(Ytrue[:, j], Ypred[:, j], labels=C4, average=None, zero_division=0) for j in range(Ytrue.shape[1])])
    return pd.DataFrame(per, index=SHORT, columns=C4) if detail else float(per.mean())


def assemble(Q, S, th, bias):
    """Q (N,9) prob relevansi, S (N,3) skor sentimen, bias (3,) atau (9,3) -> label string (N,9)."""
    bias = np.broadcast_to(np.asarray(bias, float), (9, 3))
    ls = np.log(np.clip(S, 1e-6, 1))[:, None, :] + bias[None]                 # (N,9,3)
    return np.where(Q >= th, SENT[ls.argmax(2)], "TidakRelevan")


def _f1_counts(tp, fp, fn):
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)


def tune_bias(Y, Q, S, th, per_pillar_shrink=None, grid=np.round(np.arange(-0.6, 2.41, 0.1), 2)):
    """Cari bias (Netral, Positif) yang memaksimalkan metrik resmi. per_pillar_shrink=None -> satu bias global;
    angka 0..1 -> bias per pilar, di-shrink ke global (1 = global murni)."""
    ls = np.log(np.clip(S, 1e-6, 1)); rel = Q >= th
    # F1 TidakRelevan tidak bergantung bias -> cukup optimalkan jumlah F1 tiga kelas sentimen per pilar
    A, C = np.meshgrid(grid, grid, indexing="ij"); A, C = A.ravel(), C.ravel()
    P = np.stack([np.zeros_like(A)[None] + ls[:, 0:1], ls[:, 1:2] + A[None], ls[:, 2:3] + C[None]], 2).argmax(2)   # (N, n_grid) kelas terpilih
    tot = np.zeros((9, len(A)))
    for j in range(9):
        r = rel[:, j]
        for s in range(3):
            y = (Y[:, j] == SENT[s]); p = (P == s) & r[:, None]
            tp = (p & y[:, None]).sum(0); tot[j] += _f1_counts(tp, p.sum(0) - tp, y.sum() - tp)
    g = int(tot.sum(0).argmax()); glob = np.array([0.0, A[g], C[g]])
    if per_pillar_shrink is None: return glob
    per = np.array([[0.0, A[k], C[k]] for k in tot.argmax(1)])
    return (1 - per_pillar_shrink) * per + per_pillar_shrink * glob[None]


def cv_bias_score(Y, Q, S, th, fold, per_pillar_shrink=None):
    """Estimasi jujur: bias disetel pada fold lain, diterapkan ke fold yang ditahan."""
    P = np.empty(Y.shape, dtype=object)
    for k in np.unique(fold):
        b = tune_bias(Y[fold != k], Q[fold != k], S[fold != k], th, per_pillar_shrink)
        P[fold == k] = assemble(Q[fold == k], S[fold == k], th, b)
    return score4(Y, P.astype(str))


def sent3(d, S):
    """macro-F1 3 kelas pada baris ber-sentimen tunggal & ber-caption (diagnostik kualitas model sentimen saja)."""
    m = d["OK"] & ~d["IS_X"] & (d["YS"].sum(1) == 1)
    return float(f1_score(d["YS"][m].argmax(1), S[m].argmax(1), average="macro"))


def blend(mats, weights=None):
    """rata-rata geometrik berbobot dari beberapa matriks skor sentimen (N,3)."""
    w = np.ones(len(mats)) if weights is None else np.asarray(weights, float)
    return np.exp(sum(wi * np.log(np.clip(m, 1e-6, 1)) for wi, m in zip(w, mats)) / w.sum())


def write_submission(d, Q_test, S_test, th, bias, path):
    pred = assemble(Q_test, S_test, th, bias)
    sub = pd.DataFrame(pred, columns=LABELS); sub.insert(0, "ID", d["test"].ID.values)
    smp = d["sample"]
    assert list(sub.columns) == list(smp.columns) and len(sub) == len(smp) and (sub.ID.values == smp.ID.values).all()
    assert sub.notna().all().all() and set(np.unique(pred)) <= set(C4)
    sub.to_csv(path, index=False)
    print(f"{path} tersimpan {sub.shape} | proporsi kelas test: " +
          ", ".join(f"{c} {np.mean(pred == c):.4f}" for c in C4))
    return sub


d = load_data()
train, test = d["train"], d["test"]
Y, IS_X, FOLD, TRAIN_OK = d["YS"], d["IS_X"], d["FOLD"], d["OK"]     # Y = (N,3) multi-hot sentimen
print("train", train.shape, "| test", test.shape, "| baris ber-sentimen:", int(TRAIN_OK.sum()), "| baris X tanpa caption:", int(IS_X.sum()))

train (23297, 18) | test (34687, 9) | baris ber-sentimen: 20429 | baris X tanpa caption: 1554


### 2.1 Karakteristik label sentimen

In [4]:
_rel = d["REL"]; _B = d["B"]
print("sebaran sentimen pada sel relevan:", pd.Series(_B[_rel]).value_counts(normalize=True).round(3).to_dict(), "| jumlah sel relevan:", int(_rel.sum()))
print(pd.DataFrame({SHORT[j]: pd.Series(_B[_rel[:, j], j]).value_counts() for j in range(9)}).T.assign(total=lambda t: t.sum(axis=1)).to_string())
_multi = np.where(_rel.sum(1) >= 2)[0]
print(f"baris multi-label: {len(_multi)} | sentimen identik di semua pilarnya: {np.mean([len(set(_B[i][_rel[i]])) == 1 for i in _multi]):.3f}")
_row = np.array([_B[i][_rel[i]][0] for i in np.where(TRAIN_OK)[0]])
print("sentimen per Source (baris ber-sentimen):"); print(pd.crosstab(train.Source[TRAIN_OK].values, _row, normalize="index").round(3).to_string())

sebaran sentimen pada sel relevan: {'Negatif': 0.747, 'Positif': 0.149, 'Netral': 0.104} | jumlah sel relevan: 22888
        Negatif  Netral  Positif  total
Infra      2480     132      191   2803
Kes        1648     249      677   2574
Adm        1269     220      180   1669
Ling       1171     267      555   1993
Tertib     2526     230      435   3191
Trans      1437     347      442   2226
Sos        2288     302      295   2885
Keam       2063     388      267   2718
Pend       2216     245      368   2829
baris multi-label: 2263 | sentimen identik di semua pilarnya: 0.926
sentimen per Source (baris ber-sentimen):
col_0        Negatif  Netral  Positif
row_0                                
Facebook       0.830   0.084    0.086
Instagram      0.735   0.087    0.178
TikTok         0.683   0.154    0.163
X (Twitter)    0.455   0.191    0.354
YouTube        0.789   0.079    0.132


## 3. Tahap 1 — relevansi per pilar dari babak penyisihan
Probabilitas stacker v9 (+ post-rate shrinkage) direkonstruksi dari `artifacts_final/penyisihan`; sebaran jumlah label per baris test harus sama dengan submisi penyisihan. Batas atas skor bila sentimen sempurna menunjukkan berapa yang "hilang" di tahap relevansi.

In [5]:
Q_oof, Q_test, TH_REL = relevance_v9(d)
print(f"threshold relevansi {TH_REL:.2f} | #pilar relevan per baris test: {pd.Series((Q_test >= TH_REL).sum(1)).value_counts().sort_index().to_dict()}")
_nx = ~IS_X; _Yv = d["B"][_nx]
_true_s = np.array([[np.mean(r[m] == s) if m.any() else 0.0 for s in SENT] for r, m in zip(_Yv, d["REL"][_nx])])   # sentimen sebenarnya per baris
print(f"batas atas (relevansi v9 + sentimen per-komentar sempurna): {score4(_Yv, assemble(Q_oof[_nx], _true_s + 1e-6, TH_REL, 0)):.4f}")
print(f"baseline (relevansi v9 + semua Negatif)                   : {score4(_Yv, assemble(Q_oof[_nx], np.tile([1.0, 1e-6, 1e-6], (len(_Yv), 1)), TH_REL, 0)):.4f}")

threshold relevansi 0.40 | #pilar relevan per baris test: {0: 2589, 1: 29800, 2: 2171, 3: 123, 4: 4}


batas atas (relevansi v9 + sentimen per-komentar sempurna): 0.9150


baseline (relevansi v9 + semua Negatif)                   : 0.4375


## 4. Tahap 2 (a) — sentimen TF-IDF + Logistic Regression (cadangan) dan perakitan submisi
`build_submission()` dipanggil ulang setiap kali ada model sentimen baru; ia memilih kombinasi model dan bias kelas lewat CV lalu menulis `submission.csv`.

In [6]:
def tfidf_sentiment(d):
    po, pt = f"{ART}/l1/sent_tfidf_oof.npy", f"{ART}/l1/sent_tfidf_test.npy"
    if os.path.exists(po) and os.path.exists(pt): return np.load(po), np.load(pt)
    t0 = time.time(); tr, te = d["train"], d["test"]; OK, FOLD, YS = d["OK"], d["FOLD"], d["YS"]
    xtr, xte = np.array(tr.teks_l.tolist(), dtype=object), np.array(te.teks_l.tolist(), dtype=object)
    vecs = lambda: [TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200000),
                    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=300000)]
    def fit_pred(tri, targets):
        vs = vecs(); X = hstack([v.fit_transform(xtr[tri]) for v in vs]).tocsr()
        ms = [LogisticRegression(C=4.0, max_iter=3000, solver="liblinear").fit(X, YS[tri, s]) for s in range(3)]
        return [np.column_stack([m.predict_proba(hstack([v.transform(t) for v in vs]).tocsr())[:, 1] for m in ms]) for t in targets]
    oof = np.zeros((len(tr), 3), np.float32); tst = np.zeros((len(te), 3), np.float32)
    for f in range(5):
        tri = np.where((FOLD != f) & OK)[0]; vai = np.where(FOLD == f)[0]
        a, b = fit_pred(tri, [xtr[vai], xte]); oof[vai] = a; tst += b / 5
    np.save(po, oof); np.save(pt, tst); print(f"[sent_tfidf] selesai {time.time() - t0:.0f}s | macro-F1 3 kelas {sent3(d, oof):.4f}")
    return oof, tst


def build_submission():
    # memakai semua model sentimen yang tersedia di artifacts_final/l1; blend & cara bias dipilih lewat CV (bias disetel di fold lain)
    nx = ~d["IS_X"]; Y, fold = d["B"][nx], d["FOLD"][nx]; Q, th = Q_oof[nx], TH_REL
    models = {"sent_tfidf": tfidf_sentiment(d)}
    for p in sorted(glob.glob(f"{ART}/l1/sent_*_oof.npy")):
        tag = os.path.basename(p)[:-8]
        if tag == "sent_tfidf" or not os.path.exists(f"{ART}/l1/{tag}_test.npy"): continue
        o, t = np.load(p), np.load(f"{ART}/l1/{tag}_test.npy")
        if o.shape == (len(d["train"]), 3) and np.isfinite(o[nx]).all() and np.isfinite(t).all(): models[tag] = (o, t)
        else: print(f"!! {tag} dilewati (bentuk/nilai tidak valid)")
    rows = []
    for tag, (o, t) in models.items():
        rows.append((tag, sent3(d, o), cv_bias_score(Y, Q, o[nx], th, fold), np.corrcoef(o[nx].ravel(), models["sent_tfidf"][0][nx].ravel())[0, 1]))
    print(pd.DataFrame(rows, columns=["model", "macroF1 3 kelas", "skor resmi (CV bias global)", "korelasi vs tfidf"]).round(4).to_string(index=False))
    # kandidat blend: setiap subset model neural (+ opsi tfidf berbobot 0.5); pilih CV terbaik
    neural = [k for k in models if k != "sent_tfidf"]; cands = {}
    if not neural: cands["sent_tfidf"] = (["sent_tfidf"], [1.0])
    for r in range(1, len(neural) + 1):
        for sub in itertools.combinations(neural, r):
            cands["+".join(sub)] = (list(sub), [1.0] * len(sub))
            cands["+".join(sub) + "+tfidf/2"] = (list(sub) + ["sent_tfidf"], [1.0] * len(sub) + [0.5])
    best = None; res = []
    for nm, (ks, ws) in cands.items():
        So = blend([models[k][0][nx] for k in ks], ws)
        for mode, shr in (("global", None), ("per-pilar 50%", 0.5)):
            sc = cv_bias_score(Y, Q, So, th, fold, shr); res.append((nm, mode, sc))
            if best is None or sc > best[0] + (0.0005 if shr is not None else 0): best = (sc, nm, ks, ws, shr, mode)   # per-pilar harus menang jelas
    print(pd.DataFrame(res, columns=["blend", "bias", "skor resmi CV"]).sort_values("skor resmi CV", ascending=False).head(12).round(4).to_string(index=False))
    sc, nm, ks, ws, shr, mode = best
    So, St = blend([models[k][0][nx] for k in ks], ws), blend([models[k][1] for k in ks], ws)
    bias = tune_bias(Y, Q, So, th, shr)
    P = assemble(Q, So, th, bias)
    print(f"\nTERPILIH: {nm} | bias {mode} | skor resmi CV (jujur) {sc:.4f} | in-sample {score4(Y, P):.4f}")
    print("bias [Negatif, Netral, Positif]:", np.round(bias, 2).tolist())
    det = score4(Y, P, True); det["MACRO"] = det.mean(axis=1); print(det.round(3).to_string())
    print("proporsi kelas OOF (prediksi vs sebenarnya): " + ", ".join(f"{c} {np.mean(P == c):.4f}/{np.mean(Y == c):.4f}" for c in C4))
    write_submission(d, Q_test, St, th, bias, "submission.csv")
    json.dump({"blend": ks, "weights": ws, "bias_mode": mode, "bias": np.asarray(bias).tolist(), "threshold": th, "cv": sc},
              open(f"{ART}/final_config.json", "w"), indent=1)


build_submission()

         model  macroF1 3 kelas  skor resmi (CV bias global)  korelasi vs tfidf
    sent_tfidf           0.5665                       0.6374             1.0000
      sent_ibl           0.6824                       0.6937             0.8622
sent_sailor_s1           0.7289                       0.7362             0.8628


                          blend          bias  skor resmi CV
                 sent_sailor_s1        global         0.7362
                 sent_sailor_s1 per-pilar 50%         0.7349
         sent_sailor_s1+tfidf/2        global         0.7314
         sent_sailor_s1+tfidf/2 per-pilar 50%         0.7290
        sent_ibl+sent_sailor_s1        global         0.7250
sent_ibl+sent_sailor_s1+tfidf/2        global         0.7238
sent_ibl+sent_sailor_s1+tfidf/2 per-pilar 50%         0.7225
        sent_ibl+sent_sailor_s1 per-pilar 50%         0.7219
               sent_ibl+tfidf/2        global         0.6958
               sent_ibl+tfidf/2 per-pilar 50%         0.6957
                       sent_ibl        global         0.6937
                       sent_ibl per-pilar 50%         0.6906



TERPILIH: sent_sailor_s1 | bias global | skor resmi CV (jujur) 0.7362 | in-sample 0.7387
bias [Negatif, Netral, Positif]: [0.0, 1.1, 0.9]
        TidakRelevan  Negatif  Netral  Positif  MACRO
Infra          0.973    0.795   0.444    0.440  0.663
Kes            0.996    0.887   0.540    0.874  0.824
Adm            0.990    0.831   0.656    0.664  0.786
Ling           0.987    0.797   0.371    0.789  0.736
Tertib         0.989    0.864   0.342    0.606  0.700
Trans          0.989    0.801   0.521    0.771  0.770
Sos            0.993    0.877   0.521    0.718  0.777
Keam           0.981    0.781   0.375    0.588  0.681
Pend           0.982    0.831   0.400    0.627  0.710
proporsi kelas OOF (prediksi vs sebenarnya): TidakRelevan 0.8885/0.8913, Negatif 0.0835/0.0835, Netral 0.0132/0.0107, Positif 0.0147/0.0145


submission.csv tersimpan (34687, 10) | proporsi kelas test: TidakRelevan 0.8894, Negatif 0.0789, Netral 0.0173, Positif 0.0144


## 5. Tahap 2 (b) — fine-tuning model bahasa untuk sentimen
Trainer sama dengan babak penyisihan (IndoBERT: input pasangan komentar + caption, mean pooling, multi-sample dropout; Sailor2-3B: LoRA + prefix caption di-cache, deteksi kolaps + latih ulang). Perbedaannya: keluaran 3 sigmoid sentimen, hanya baris ber-sentimen yang dilatih, dan ambang kolaps disesuaikan dengan skala loss sentimen (BCE prior ±0,43).

In [7]:
GRID = np.round(np.arange(0.02, 0.981, 0.01), 3)

def f1_curve(y, p, grid=GRID):
    # binary macro-F1 (rata-rata F1 kelas 0 & kelas 1) untuk setiap threshold pada grid (vektorisasi)
    y = y.astype(bool); P = p[None, :] >= grid[:, None]
    tp = (P & y).sum(1); fp = (P & ~y).sum(1); fn = (~P & y).sum(1); tn = (~P & ~y).sum(1)
    return 0.5 * (2 * tp / np.maximum(2 * tp + fp + fn, 1) + 2 * tn / np.maximum(2 * tn + fn + fp, 1))

def per_label(Y, P, th=0.5):
    th = np.broadcast_to(np.asarray(th, float), (Y.shape[1],))
    return np.array([f1_curve(Y[:, j], P[:, j], np.array([th[j]]))[0] for j in range(Y.shape[1])])

def score(Y, P, th=0.5):
    # metrik kompetisi: Mean Column-wise Macro-F1
    return float(per_label(Y, P, th).mean())

def tune(Y, P, shrink=0.0):
    # threshold per-label, di-shrink ke satu threshold global (lebih stabil untuk data ber-cluster per postingan)
    per = np.array([GRID[int(np.argmax(f1_curve(Y[:, j], P[:, j])))] for j in range(Y.shape[1])])
    glob = GRID[int(np.argmax(np.mean([f1_curve(Y[:, j], P[:, j]) for j in range(Y.shape[1])], 0)))]
    return (1 - shrink) * per + shrink * glob

def cv_thr_score(Y, P, fold, shrink=0.5):
    # estimasi JUJUR: threshold di-tuning pada OOF fold lain lalu diterapkan ke fold yang ditahan
    pred = np.zeros_like(Y)
    for k in np.unique(fold):
        th = tune(Y[fold != k], P[fold != k], shrink)
        pred[fold == k] = (P[fold == k] >= th).astype(int)
    return score(Y, pred.astype(float), 0.5)

def report(name, P, mask=None):
    mask = (~IS_X & TRAIN_OK) if mask is None else mask
    Yv, Pv, fv = Y[mask], P[mask], FOLD[mask]
    return {"model": name, "F1@0.5": score(Yv, Pv), "F1@tuned(OOF)": score(Yv, Pv, tune(Yv, Pv)),
            "F1@cv-thr": cv_thr_score(Yv, Pv, fv, 0.0), "F1@cv-thr-shrink": cv_thr_score(Yv, Pv, fv, 0.5),
            "F1@cv-thr-global": cv_thr_score(Yv, Pv, fv, 1.0)}

RESULTS = []

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Windows (WDDM) tidak memunculkan OOM saat VRAM penuh, melainkan diam-diam "spill" ke RAM sistem (training >10x lebih lambat).
# Membatasi allocator ke 88% VRAM membuat PyTorch membebaskan cache-nya terlebih dahulu alih-alih spill.
if DEV.type == "cuda": torch.cuda.set_per_process_memory_fraction(0.88)

class Net(nn.Module):
    def __init__(self, name, n_out=NOUT, msd=5, drop=0.2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name, add_pooling_layer=False, attn_implementation="sdpa")
        self.drops = nn.ModuleList([nn.Dropout(drop) for _ in range(msd)])
        self.head = nn.Linear(self.backbone.config.hidden_size, n_out)
    def forward(self, ids, am, tt=None):
        kw = {"input_ids": ids, "attention_mask": am}
        if tt is not None: kw["token_type_ids"] = tt
        h = self.backbone(**kw).last_hidden_state
        m = am.unsqueeze(-1).to(h.dtype); z = (h * m).sum(1) / m.sum(1).clamp(min=1.0)       # mean pooling
        if self.training: return torch.stack([self.head(d(z)) for d in self.drops]).mean(0)  # multi-sample dropout
        return self.head(z)

class PairEncoder:
    # komentar lebih dulu (maks max_comment token), caption mengisi sisa hingga max_len
    def __init__(self, tok, max_len=256, max_comment=160):
        self.tok, self.max_len, self.max_comment = tok, max_len, max_comment
        self.n_special = tok.num_special_tokens_to_add(pair=True)
    def __call__(self, comments, captions):
        A = self.tok(list(comments), add_special_tokens=False)["input_ids"]
        uniq = list(dict.fromkeys(captions)); cap_ids = dict(zip(uniq, self.tok(uniq, add_special_tokens=False)["input_ids"]))
        ids, tts = [], []
        for a, c in zip(A, captions):
            budget = self.max_len - self.n_special
            a = a[:min(len(a), self.max_comment, budget)]; b = cap_ids[c][:max(0, budget - len(a))]
            ids.append(self.tok.build_inputs_with_special_tokens(a, b)); tts.append(self.tok.create_token_type_ids_from_sequences(a, b))
        return {"input_ids": ids, "token_type_ids": tts}

def make_batches(idx, enc, bs, rng=None):
    lens = np.array([len(enc["input_ids"][i]) for i in idx])
    if rng is None:
        order = idx[np.argsort(lens, kind="stable")]; return [order[i:i + bs] for i in range(0, len(order), bs)]
    perm = rng.permutation(len(idx)); out = []
    for k in range(0, len(perm), bs * 32):
        mb = perm[k:k + bs * 32]; mb = mb[np.argsort(lens[mb], kind="stable")]
        out += [idx[mb[j:j + bs]] for j in range(0, len(mb), bs)]
    rng.shuffle(out); return out

def collate(ch, enc, pad_id, use_tt, alt=None, alt_mask=None):
    rows = [((alt if (alt is not None and alt_mask[k]) else enc)["input_ids"][i], (alt if (alt is not None and alt_mask[k]) else enc)["token_type_ids"][i]) for k, i in enumerate(ch)]
    L = max(len(r[0]) for r in rows)
    ids = torch.full((len(rows), L), pad_id, dtype=torch.long); am = torch.zeros((len(rows), L), dtype=torch.long); tt = torch.zeros((len(rows), L), dtype=torch.long)
    for k, (a, t) in enumerate(rows): ids[k, :len(a)] = torch.tensor(a); am[k, :len(a)] = 1; tt[k, :len(t)] = torch.tensor(t)
    return ids.to(DEV), am.to(DEV), (tt.to(DEV) if use_tt else None)

@torch.no_grad()
def predict(model, idx, enc, bs, pad_id, use_tt):
    model.eval(); idx = np.asarray(idx); out = np.zeros((len(idx), NOUT), np.float32); pos = {v: k for k, v in enumerate(idx)}
    with torch.autocast("cuda", dtype=torch.bfloat16):
        for ch in make_batches(idx, enc, bs):
            ids, am, tt = collate(ch, enc, pad_id, use_tt)
            out[[pos[i] for i in ch]] = torch.sigmoid(model(ids, am, tt).float()).cpu().numpy()
    return out

def tlog(msg):
    # cetak ke output sel sekaligus ke artifacts/train_log.txt (memantau progres saat run-all panjang)
    print(msg, flush=True)
    with open(f"{ART}/train_log.txt", "a", encoding="utf-8") as fh: fh.write(msg + chr(10))

def train_backbone(tag, model_name, cased, lr, bs, epochs=2, freeze_emb=False, grad_ckpt=False, head_lr=5e-4,
                   max_len=256, max_comment=160, cap_drop=0.1, wd=0.01, warmup=0.1, seed=SEED):
    t_all = time.time()
    tok = AutoTokenizer.from_pretrained(model_name); penc = PairEncoder(tok, max_len, max_comment)
    tc, cc = ("teks_c", "cap_c") if cased else ("teks_l", "cap_l")
    ttr, ctr, tte, cte = train[tc].tolist(), train[cc].tolist(), test[tc].tolist(), test[cc].tolist()
    enc_tr, enc_tr_nocap, enc_te = penc(ttr, ctr), penc(ttr, [""] * len(ttr)), penc(tte, cte)
    ucap_tr, ucap_te = sorted(set(c for c in ctr if c)), sorted(set(cte))
    enc_cap_tr, enc_cap_te = penc([""] * len(ucap_tr), ucap_tr), penc([""] * len(ucap_te), ucap_te)
    cap_row_tr = {c: k for k, c in enumerate(ucap_tr)}; cap_row_te = {c: k for k, c in enumerate(ucap_te)}
    use_tt = "roberta" not in model_name.lower(); pad = tok.pad_token_id
    pbs = 64 if bs >= 32 else 32                     # batch inferensi (lebih kecil untuk model large -> hindari VRAM spill)
    oof = np.full((len(train), NOUT), np.nan, np.float32); oof_cap = np.full((len(train), NOUT), np.nan, np.float32)
    tst = np.zeros((len(test), NOUT), np.float32); tst_cap = np.zeros((len(ucap_te), NOUT), np.float32)
    os.makedirs(f"{ART}/models/{tag}", exist_ok=True)
    tlog(f"[{tag}] {model_name} | cased={cased} lr={lr} bs={bs} epochs={epochs} freeze_emb={freeze_emb}")
    for f in range(5):
        seed_all(seed + f); rng = np.random.default_rng(seed + f)
        tri, vai = np.where((FOLD != f) & TRAIN_OK)[0], np.where(FOLD == f)[0]; vnx = vai[~IS_X[vai] & TRAIN_OK[vai]]   # final: latih & validasi hanya baris ber-sentimen
        model = Net(model_name).to(DEV)
        if freeze_emb: model.backbone.get_input_embeddings().weight.requires_grad_(False)
        if grad_ckpt: model.backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        nd = ["bias", "LayerNorm.weight", "LayerNorm.bias", "layer_norm"]
        bb = [(n, p) for n, p in model.backbone.named_parameters() if p.requires_grad]
        groups = [{"params": [p for n, p in bb if not any(k in n for k in nd)], "lr": lr, "weight_decay": wd},
                  {"params": [p for n, p in bb if any(k in n for k in nd)], "lr": lr, "weight_decay": 0.0},
                  {"params": list(model.head.parameters()), "lr": head_lr, "weight_decay": 0.0}]
        opt = torch.optim.AdamW(groups, lr=lr, eps=1e-6, fused=True)
        spe = math.ceil(len(tri) / bs); sch = get_linear_schedule_with_warmup(opt, int(warmup * spe * epochs), spe * epochs)
        crit, Yt = nn.BCEWithLogitsLoss(), torch.from_numpy(Y.astype(np.float32)); torch.cuda.reset_peak_memory_stats()
        for ep in range(epochs):
            model.train(); t0 = time.time(); tot = 0.0
            for bi, ch in enumerate(make_batches(tri, enc_tr, bs, rng)):
                ids, am, tt = collate(ch, enc_tr, pad, use_tt, enc_tr_nocap, rng.uniform(size=len(ch)) < cap_drop)
                with torch.autocast("cuda", dtype=torch.bfloat16): logits = model(ids, am, tt)
                loss = crit(logits.float(), Yt[ch].to(DEV)); loss.backward()
                torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], 1.0)
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True); tot += loss.item()
                if bi % 400 == 0: tlog(f"   fold {f} ep {ep + 1} step {bi:4d}/{spe} | loss {tot / (bi + 1):.4f} | lr {sch.get_last_lr()[0]:.2e}")
            torch.cuda.empty_cache(); pv = predict(model, vnx, enc_tr, pbs, pad, use_tt); yv = Y[vnx]; torch.cuda.empty_cache()
            tlog(f"  fold {f} | epoch {ep + 1}/{epochs} | train_bce {tot / (bi + 1):.4f} | val F1@0.5 {score(yv, pv):.4f} | "
                  f"val F1@tuned {score(yv, pv, tune(yv, pv)):.4f} | {time.time() - t0:.0f}s | peak mem {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")
        oof[vai] = predict(model, vai, enc_tr, pbs, pad, use_tt)
        vcaps = sorted(set(ctr[i] for i in vai if ctr[i]))
        cmap = dict(zip(vcaps, predict(model, [cap_row_tr[c] for c in vcaps], enc_cap_tr, pbs, pad, use_tt)))
        for i in vai:
            if ctr[i]: oof_cap[i] = cmap[ctr[i]]
        tst += predict(model, np.arange(len(test)), enc_te, pbs, pad, use_tt) / 5
        tst_cap += predict(model, np.arange(len(ucap_te)), enc_cap_te, pbs, pad, use_tt) / 5
        torch.save({k: v.half() for k, v in model.state_dict().items()}, f"{ART}/models/{tag}/fold{f}.pth")
        del model, opt, sch, groups, bb, logits, loss, ids, am, tt; gc.collect(); torch.cuda.empty_cache()
    out = {"oof": oof, "test": tst, "oof_cap": oof_cap, "test_cap": tst_cap[[cap_row_te[c] for c in cte]]}
    for k, v in out.items(): np.save(f"{ART}/l1/{tag}_{k}.npy", v)
    r = report(tag, oof); RESULTS.append(r)
    tlog(f"[{tag}] OOF(nonX) F1@tuned {r['F1@tuned(OOF)']:.4f} | cv-thr-shrink {r['F1@cv-thr-shrink']:.4f} | total {(time.time() - t_all) / 60:.1f} min")
    return out


def _done(tag):
    return all(os.path.exists(f"{ART}/l1/{tag}_{k}.npy") for k in ("oof", "test"))

def _save_tag_log(tag):
    # salinan log pelatihan model ini (ditampilkan lagi bila nanti dimuat dari artifacts_final); trainer Sailor menulisnya sendiri
    lp = f"{ART}/l1/{tag}_log.txt"
    if os.path.exists(lp): return
    lines = open(f"{ART}/train_log.txt", encoding="utf-8").read().split(chr(10))
    starts = [i for i, l in enumerate(lines) if l.startswith(f"[{tag}] ") and "OOF(nonX)" not in l]
    if starts: open(lp, "w", encoding="utf-8").write(chr(10).join(lines[starts[-1]:]).rstrip() + chr(10))

def run_model(tag, kind, seed=SEED):
    # melatih satu model sentimen (dimuat dari artifacts_final bila sudah ada), lalu merakit ulang submisi
    if _done(tag):
        print(f"[{tag}] sudah ada di {ART}/l1 -> dimuat, tidak dilatih ulang. Log pelatihan aslinya:")
        if os.path.exists(f"{ART}/l1/{tag}_log.txt"): print(open(f"{ART}/l1/{tag}_log.txt", encoding="utf-8").read().rstrip())
    elif not torch.cuda.is_available():
        print(f"[{tag}] dilewati: tidak ada GPU CUDA"); return
    else:
        try:
            if kind == "ibl":
                train_backbone(tag, IBL_NAME, cased=False, lr=2e-5, bs=16, epochs=2, seed=seed)
            else:
                train_llm_backbone(tag, LLM_NAME, bs=16, tok_budget=1024, max_cap=160, grad_ckpt=True, seed=seed, max_retries=1,
                                   collapse_tail=0.41, collapse_f1=0.55, abort_loss=0.42)
            _save_tag_log(tag)
        except Exception:
            tlog(f"!! [{tag}] gagal:" + chr(10) + traceback.format_exc()); return
    build_submission()


### 5.1 IndoBERT-large-p2

In [8]:
run_model("sent_ibl", "ibl")

[sent_ibl] sudah ada di artifacts_final/l1 -> dimuat, tidak dilatih ulang. Log pelatihan aslinya:
[sent_ibl] indobenchmark/indobert-large-p2 | cased=False lr=2e-05 bs=16 epochs=2 freeze_emb=False
   fold 0 ep 1 step    0/1017 | loss 0.8351 | lr 9.85e-08
   fold 0 ep 1 step  400/1017 | loss 0.3656 | lr 1.78e-05
   fold 0 ep 1 step  800/1017 | loss 0.3170 | lr 1.35e-05
  fold 0 | epoch 1/2 | train_bce 0.3055 | val F1@0.5 0.7678 | val F1@tuned 0.7838 | 182s | peak mem 9.5 GB
   fold 0 ep 2 step    0/1017 | loss 0.3071 | lr 1.11e-05
   fold 0 ep 2 step  400/1017 | loss 0.1715 | lr 6.73e-06
   fold 0 ep 2 step  800/1017 | loss 0.1649 | lr 2.36e-06
  fold 0 | epoch 2/2 | train_bce 0.1632 | val F1@0.5 0.7772 | val F1@tuned 0.7929 | 183s | peak mem 10.7 GB
   fold 1 ep 1 step    0/1026 | loss 0.7342 | lr 9.76e-08
   fold 1 ep 1 step  400/1026 | loss 0.3556 | lr 1.79e-05
   fold 1 ep 1 step  800/1026 | loss 0.3046 | lr 1.35e-05
  fold 1 | epoch 1/2 | train_bce 0.2942 | val F1@0.5 0.7528 | val F

         model  macroF1 3 kelas  skor resmi (CV bias global)  korelasi vs tfidf
    sent_tfidf           0.5665                       0.6374             1.0000
      sent_ibl           0.6824                       0.6937             0.8622
sent_sailor_s1           0.7289                       0.7362             0.8628


                          blend          bias  skor resmi CV
                 sent_sailor_s1        global         0.7362
                 sent_sailor_s1 per-pilar 50%         0.7349
         sent_sailor_s1+tfidf/2        global         0.7314
         sent_sailor_s1+tfidf/2 per-pilar 50%         0.7290
        sent_ibl+sent_sailor_s1        global         0.7250
sent_ibl+sent_sailor_s1+tfidf/2        global         0.7238
sent_ibl+sent_sailor_s1+tfidf/2 per-pilar 50%         0.7225
        sent_ibl+sent_sailor_s1 per-pilar 50%         0.7219
               sent_ibl+tfidf/2        global         0.6958
               sent_ibl+tfidf/2 per-pilar 50%         0.6957
                       sent_ibl        global         0.6937
                       sent_ibl per-pilar 50%         0.6906



TERPILIH: sent_sailor_s1 | bias global | skor resmi CV (jujur) 0.7362 | in-sample 0.7387
bias [Negatif, Netral, Positif]: [0.0, 1.1, 0.9]
        TidakRelevan  Negatif  Netral  Positif  MACRO
Infra          0.973    0.795   0.444    0.440  0.663
Kes            0.996    0.887   0.540    0.874  0.824
Adm            0.990    0.831   0.656    0.664  0.786
Ling           0.987    0.797   0.371    0.789  0.736
Tertib         0.989    0.864   0.342    0.606  0.700
Trans          0.989    0.801   0.521    0.771  0.770
Sos            0.993    0.877   0.521    0.718  0.777
Keam           0.981    0.781   0.375    0.588  0.681
Pend           0.982    0.831   0.400    0.627  0.710


proporsi kelas OOF (prediksi vs sebenarnya): TidakRelevan 0.8885/0.8913, Negatif 0.0835/0.0835, Netral 0.0132/0.0107, Positif 0.0147/0.0145
submission.csv tersimpan (34687, 10) | proporsi kelas test: TidakRelevan 0.8894, Negatif 0.0789, Netral 0.0173, Positif 0.0144


### 5.2 Trainer Sailor2-3B — LoRA dengan prefix caching (bf16)

In [9]:
import contextlib, functools, hashlib
from torch.utils.checkpoint import checkpoint as _torch_ckpt
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
LLM_CACHE_DIR = os.environ.get("LLM_CACHE_DIR")            # lokasi cache HF untuk bobot LLM besar (opsional)
LLM_TRAIN_META = globals().get("LLM_TRAIN_META", {})        # ringkasan percobaan per fold (juga ditulis ke {ART}/l1/<tag>_train_meta.json)

def _on_cuda():
    return DEV.type == "cuda"

def _ac():                                                     # autocast bf16 di GPU; no-op di CPU
    return torch.autocast("cuda", dtype=torch.bfloat16) if _on_cuda() else contextlib.nullcontext()

try:                                                           # transformers >= 4.54: cache = daftar layer (key_cache/value_cache dihapus)
    from transformers.cache_utils import DynamicLayer
except ImportError:                                            # transformers lama: cache = key_cache/value_cache
    DynamicLayer = None

if DynamicLayer is not None:
    class _PrefixLayer(DynamicLayer):
        # layer KV prefix read-only: update() mengembalikan cat(prefix, baru) TANPA menyimpan
        def __init__(self, k, v):
            super().__init__(); self.keys, self.values, self.is_initialized = k, v, True; self.dtype, self.device = k.dtype, k.device
        def update(self, key_states, value_states, cache_kwargs=None):
            return torch.cat([self.keys, key_states], dim=-2), torch.cat([self.values, value_states], dim=-2)

class PrefixCache(DynamicCache):
    # KV prefix read-only: update() mengembalikan cat(prefix, baru) TANPA menyimpan -> aman untuk recompute gradient checkpointing
    def update(self, key_states, value_states, layer_idx, cache_kwargs=None):
        if DynamicLayer is not None:
            return self.layers[layer_idx].update(key_states, value_states, cache_kwargs)
        return (torch.cat([self.key_cache[layer_idx], key_states], dim=-2), torch.cat([self.value_cache[layer_idx], value_states], dim=-2))

def _kv_pairs(pkv):                                            # [(keys, values)] per layer, lintas versi transformers
    if hasattr(pkv, "layers"): return [(l.keys, l.values) for l in pkv.layers]
    if hasattr(pkv, "key_cache"): return list(zip(pkv.key_cache, pkv.value_cache))
    return [(x[0], x[1]) for x in pkv]

def expand_cache(pkv, B):
    pairs = _kv_pairs(pkv)
    c = PrefixCache()
    if DynamicLayer is not None:
        c.layers = [_PrefixLayer(k.expand(B, -1, -1, -1), v.expand(B, -1, -1, -1)) for k, v in pairs]
    else:
        for k, v in pairs:
            c.key_cache.append(k.expand(B, -1, -1, -1)); c.value_cache.append(v.expand(B, -1, -1, -1))
        if hasattr(c, "_seen_tokens"): c._seen_tokens = pairs[0][0].shape[-2]
    return c

class LLMNet(nn.Module):
    def __init__(self, name, r=16, alpha=32, drop=0.1, load_4bit=False, bidir=False):
        super().__init__()
        kw = dict(torch_dtype=torch.bfloat16 if _on_cuda() else torch.float32, attn_implementation="sdpa", cache_dir=LLM_CACHE_DIR)
        if load_4bit:
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                                                           bnb_4bit_compute_dtype=torch.bfloat16)
            kw["device_map"] = {"": 0}
        base = AutoModel.from_pretrained(name, **kw)
        cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=0.05, bias="none",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
        self.backbone = get_peft_model(base, cfg)
        for p in self.backbone.parameters():
            if p.requires_grad: p.data = p.data.float()            # adapter LoRA fp32, bobot dasar beku
        self.drop = nn.Dropout(drop); self.head = nn.Linear(base.config.hidden_size, NOUT)
        self._ckpt, self._ckpt_on, self.bidir = False, False, bidir
    def enable_ckpt(self):
        # Checkpointing per layer dibungkus sendiri (BUKAN gradient_checkpointing_enable HF): pada transformers baru, layer yang
        # di-checkpoint (GradientCheckpointingLayer) membuang past_key_values & use_cache -> prefix caption hilang saat training.
        # Ekuivalen dengan perilaku HF lama: checkpoint(layer, hidden_states, ..., use_reentrant=False), past_key_values tetap diteruskan.
        if self._ckpt: return
        for layer in self.backbone.get_base_model().layers:
            def fwd(hidden_states, *a, _orig=layer.forward, _layer=layer, **kw):
                if self._ckpt_on and _layer.training:
                    return _torch_ckpt(functools.partial(_orig, **kw), hidden_states, *a, use_reentrant=False)
                return _orig(hidden_states, *a, **kw)
            layer.forward = fwd
        self._ckpt = self._ckpt_on = True
    def prefix(self, pids):                                        # prefix tidak di-checkpoint (butuh cache sungguhan)
        was = self._ckpt_on; self._ckpt_on = False
        try:
            return self.backbone(input_ids=pids, attention_mask=torch.ones_like(pids), use_cache=True).past_key_values
        finally:
            self._ckpt_on = was
    def forward(self, pkv, P, ids, am):
        B, Lc = ids.shape
        pos = (P + torch.arange(Lc, device=ids.device)).unsqueeze(0).expand(B, -1)
        if self.bidir:                                             # mask aditif 4D (B,1,Lc,P+Lc): semua token prefix + semua token komentar non-pad
            dt = self.backbone.get_base_model().embed_tokens.weight.dtype
            ok = torch.cat([torch.ones(B, P, dtype=torch.bool, device=am.device), am.bool()], 1)[:, None, None, :].expand(B, 1, Lc, P + Lc)
            full_am = torch.zeros(B, 1, Lc, P + Lc, dtype=dt, device=am.device).masked_fill(~ok, torch.finfo(dt).min)
        else:
            full_am = torch.cat([torch.ones(B, P, dtype=am.dtype, device=am.device), am], 1)
        h = self.backbone(input_ids=ids, attention_mask=full_am, position_ids=pos, past_key_values=expand_cache(pkv, B),
                          use_cache=not (self.training and self._ckpt)).last_hidden_state
        return self.head(self.drop(h[torch.arange(B, device=ids.device), am.sum(1) - 1].float()))

class LLMEnc:
    def __init__(self, tok, max_cap=160, max_comment=160):
        self.tok = tok; e = lambda x: tok(x, add_special_tokens=False)["input_ids"]
        self.p_ctx, self.p_kom = e("Konteks postingan: "), e(chr(10) + "Komentar: ")
        self.bos = [tok.bos_token_id] if (tok.bos_token_id is not None and tok.bos_token_id != tok.eos_token_id) else []
        self.eos, self.max_cap, self.max_comment = tok.eos_token_id, max_cap, max_comment
    def prefixes(self, caps):
        u = list(dict.fromkeys(caps)); ids = self.tok(u, add_special_tokens=False)["input_ids"]
        return {c: self.bos + self.p_ctx + x[:self.max_cap] + self.p_kom for c, x in zip(u, ids)}
    def comments(self, texts):
        return [x[:self.max_comment] + [self.eos] for x in self.tok(list(texts), add_special_tokens=False)["input_ids"]]

def pad_rows(rows, pad_id):
    L = max(len(r) for r in rows)
    ids = torch.full((len(rows), L), pad_id, dtype=torch.long); am = torch.zeros((len(rows), L), dtype=torch.long)
    for k, a in enumerate(rows): ids[k, :len(a)] = torch.tensor(a); am[k, :len(a)] = 1
    return ids.to(DEV), am.to(DEV)

@torch.no_grad()
def predict_grouped(model, keys, rows_idx, com, pre, pad_id, bs=64, budget=4096):
    # keys[r] = kunci prefix (caption) baris r; prefix dihitung sekali per kunci, komentar diprediksi per potongan (anggaran token)
    model.eval(); out = np.zeros((len(rows_idx), NOUT), np.float32); pos = {r: k for k, r in enumerate(rows_idx)}; by = {}
    for r in rows_idx: by.setdefault(keys[r], []).append(r)
    with _ac():
        for key, rs in by.items():
            pids = torch.tensor([pre[key]], device=DEV); pkv = model.prefix(pids); P = pids.shape[1]
            rs = sorted(rs, key=lambda r: len(com[r])); chunks, cur = [], []
            for r in rs:
                if cur and (len(cur) >= bs or (len(cur) + 1) * len(com[r]) > budget): chunks.append(cur); cur = []
                cur.append(r)
            if cur: chunks.append(cur)
            for ch in chunks:
                ids, am = pad_rows([com[r] for r in ch], pad_id)
                out[[pos[r] for r in ch]] = torch.sigmoid(model(pkv, P, ids, am).float()).cpu().numpy()
    return out

def _keep_key(k):                                                  # bagian state_dict yang dilatih / disimpan
    return "lora_" in k or k.startswith("head.")

def _parse_force(spec):                                            # hook uji: LLM_FORCE_COLLAPSE="all" atau "fold:attempt,..." (refit: "full-data:0")
    spec = (spec or "").strip()
    if not spec: return lambda f, a: False
    if spec.lower() in ("1", "all"): return lambda f, a: a == 0
    pairs = {tuple(t.split(":")) for t in spec.split(",") if ":" in t}
    return lambda f, a: (str(f), str(a)) in pairs

def _fold_md5():
    return hashlib.md5(np.ascontiguousarray(FOLD, dtype=np.int64).tobytes()).hexdigest()

def _load_cached(tag, cfg):
    # hasil sel train_llm_backbone yang sama dari run sebelumnya: dipakai hanya bila konfigurasi & fold identik dan array lengkap/valid
    parts, mp = ["oof", "test", "oof_cap", "test_cap"], f"{ART}/l1/{tag}_train_meta.json"
    if not (globals().get("REUSE_L1", True) and os.path.exists(mp) and all(os.path.exists(f"{ART}/l1/{tag}_{k}.npy") for k in parts)):
        return None
    meta = json.load(open(mp, encoding="utf-8"))
    if meta.get("config") != cfg or meta.get("fold_md5") != _fold_md5():
        tlog(f"[{tag}] cache diabaikan (konfigurasi/fold berbeda) -> dilatih ulang"); return None
    out = {k: np.load(f"{ART}/l1/{tag}_{k}.npy") for k in parts}
    if cfg.get("full_refit") and all(os.path.exists(f"{ART}/l1/{tag}_{k}_full.npy") for k in ["test", "test_cap"]):
        out["test_full"], out["test_cap_full"] = np.load(f"{ART}/l1/{tag}_test_full.npy"), np.load(f"{ART}/l1/{tag}_test_cap_full.npy")
    if not (out["oof"].shape == (len(train), NOUT) and out["test"].shape == (len(test), NOUT) and np.isfinite(out["oof"]).all() and np.isfinite(out["test"]).all()):
        tlog(f"[{tag}] cache rusak -> dilatih ulang"); return None
    LLM_TRAIN_META[tag] = meta
    lp = f"{ART}/l1/{tag}_log.txt"
    if os.path.exists(lp): print(open(lp, encoding="utf-8").read().rstrip())   # log training asli run tersebut
    r = report(tag, out["oof"]); RESULTS.append(r)
    tlog(f"[{tag}] dimuat dari cache {ART}/l1 (konfigurasi & fold identik) | OOF(nonX) F1@tuned {r['F1@tuned(OOF)']:.4f} | "
         f"cv-thr-global {r['F1@cv-thr-global']:.4f} | refit full-data {'ada' if 'test_full' in out else 'tidak ada'}")
    return out

def train_llm_backbone(tag, model_name, lr=2e-4, head_lr=1e-3, bs=16, tok_budget=1024, max_cap=160, epochs=1, wd=0.01, warmup=0.1,
                       cap_drop=0.1, load_4bit=False, grad_ckpt=True, seed=SEED, full_refit=False, max_retries=2, bidir=False,
                       collapse_tail=0.25, collapse_std=0.08, collapse_f1=0.85, abort_at=0.5, abort_loss=0.30):
    cfg = dict(model=model_name, lr=lr, head_lr=head_lr, bs=bs, tok_budget=tok_budget, max_cap=max_cap, epochs=epochs,
               seed=seed, full_refit=full_refit, max_retries=max_retries)
    if bidir: cfg["bidir"] = True                                  # kunci hanya ditambahkan bila aktif -> cache run v8 (tanpa kunci) tetap cocok
    cached = _load_cached(tag, cfg)
    if cached is not None: return cached
    if globals().get("REQUIRE_CACHE", False): raise RuntimeError(f"[{tag}] cache tidak ditemukan (REQUIRE_CACHE)")
    t_all = time.time()
    tok = AutoTokenizer.from_pretrained(model_name, cache_dir=LLM_CACHE_DIR); enc = LLMEnc(tok, max_cap)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    ctr, cte = train["cap_c"].tolist(), test["cap_c"].tolist()
    com_tr, com_te = enc.comments(train["teks_c"].tolist()), enc.comments(test["teks_c"].tolist())
    pre = enc.prefixes(ctr + cte + [""])
    K = int(np.max(FOLD)) + 1
    oof = np.full((len(train), NOUT), np.nan, np.float32); oof_cap = np.full((len(train), NOUT), np.nan, np.float32)
    tst = np.zeros((len(test), NOUT), np.float32); tst_cap = np.zeros((len(test), NOUT), np.float32)
    ucap_te = sorted(set(cte)); os.makedirs(f"{ART}/models/{tag}", exist_ok=True)
    forced, n_att = _parse_force(os.environ.get("LLM_FORCE_COLLAPSE")), 1 + max_retries
    meta = {"tag": tag, "config": cfg, "fold_md5": _fold_md5(), "folds": {}}
    tlog(f"[{tag}] {model_name} | QLoRA4bit={load_4bit} r=16 | lr={lr} bs<={bs} tok_budget={tok_budget} epochs={epochs} | "
         f"komentar {np.mean([len(x) for x in com_tr]):.1f} tok, prefix {np.mean([len(pre[c]) for c in ctr]):.1f} tok/baris | full_refit={full_refit} bidir={bidir}")

    def train_one(name, att, tri, vnx, s, lr_a, hlr_a):
        # satu percobaan training; vnx = baris validasi (None untuk refit full-data -> cek in-sample pada sampel baris latih)
        seed_all(s); rng = np.random.default_rng(s)
        model = LLMNet(model_name, load_4bit=load_4bit, bidir=bidir); model = model.cuda() if load_4bit else model.to(DEV)
        if grad_ckpt: model.enable_ckpt()
        lora, head_p = [p for p in model.backbone.parameters() if p.requires_grad], list(model.head.parameters())
        groups = [{"params": lora, "lr": lr_a, "weight_decay": wd}, {"params": head_p, "lr": hlr_a, "weight_decay": 0.0}]
        opt = torch.optim.AdamW(groups, lr=lr_a, eps=1e-6)
        def make_micro():
            keys = np.array([ctr[i] if (ctr[i] and rng.uniform() >= cap_drop) else "" for i in tri], dtype=object)   # caption dropout
            mbs = []
            for key in pd.unique(keys):
                rs = tri[keys == key]; rs = rs[rng.permutation(len(rs))]; cur, mx = [], 0
                for r in rs:
                    m2 = max(mx, len(com_tr[r]))
                    if cur and (len(cur) >= bs or (len(cur) + 1) * m2 > tok_budget): mbs.append((key, np.array(cur))); cur, m2 = [], len(com_tr[r])
                    cur.append(r); mx = m2
                if cur: mbs.append((key, np.array(cur)))
            return [mbs[i] for i in rng.permutation(len(mbs))]
        n_mb = len(make_micro()); sch = get_linear_schedule_with_warmup(opt, int(warmup * n_mb * epochs), n_mb * epochs)
        crit, Yt = nn.BCEWithLogitsLoss(), torch.from_numpy(Y.astype(np.float32))
        if _on_cuda(): torch.cuda.reset_peak_memory_stats()
        info = dict(attempt=att, seed=int(s), lr=float(lr_a), head_lr=float(hlr_a), n_mb=int(n_mb), aborted=False, reason="")
        abort_bi, t_f = (int(abort_at * n_mb) - 1 if (att < n_att - 1 and abort_at) else -1), time.time()
        for ep in range(epochs):
            model.train(); t0 = time.time(); tot, nseen, mb_l, mb_n = 0.0, 0, [], []
            for bi, (key, ch) in enumerate(make_micro()):
                pids = torch.tensor([pre[key]], device=DEV); ids, am = pad_rows([com_tr[i] for i in ch], pad)
                with _ac(): logits = model(model.prefix(pids), pids.shape[1], ids, am)
                loss = crit(logits.float(), Yt[ch].to(DEV)); (loss * len(ch) / bs).backward()
                torch.nn.utils.clip_grad_norm_(lora + head_p, 1.0)
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
                lv = loss.item(); tot += lv * len(ch); nseen += len(ch); mb_l.append(lv * len(ch)); mb_n.append(len(ch))
                if bi % 400 == 0: tlog(f"   {name} ep {ep + 1} micro-batch {bi:5d}/{n_mb} | loss {tot / max(nseen, 1):.4f}")
                if not math.isfinite(lv):
                    info.update(aborted=True, reason=f"loss non-finite pada micro-batch {bi}"); break
                if ep == 0 and bi == abort_bi:                     # masih di level prior pada pertengahan epoch -> hentikan lebih awal
                    a0 = max(0, int((abort_at - 0.15) * n_mb)); w = sum(mb_l[a0:]) / max(sum(mb_n[a0:]), 1)
                    if w > abort_loss: info.update(aborted=True, reason=f"loss micro-batch {a0}-{bi} = {w:.3f} > {abort_loss} (level prior)"); break
            if info["aborted"]:
                tlog(f"  {name} | percobaan {att} DIHENTIKAN di micro-batch {bi}/{n_mb}: {info['reason']}"); break
            if _on_cuda(): torch.cuda.empty_cache()
            k4 = max(1, len(mb_l) // 4); tail = sum(mb_l[-k4:]) / max(sum(mb_n[-k4:]), 1)
            ev = vnx if vnx is not None else tri[~IS_X[tri]][::7]       # refit full-data: cek in-sample pada sampel baris LATIH
            pv = predict_grouped(model, ctr, list(ev), com_tr, pre, pad); yv = Y[ev]; f05, ft = score(yv, pv), score(yv, pv, tune(yv, pv))
            sd, lab = float(pv.std(0).mean()), ("val" if vnx is not None else "latih(in-sample)")
            tlog(f"  {name} | epoch {ep + 1}/{epochs} | train_bce {tot / max(nseen, 1):.4f} | {lab} F1@0.5 {f05:.4f} | {lab} F1@tuned {ft:.4f} | "
                 f"{time.time() - t0:.0f}s | peak mem {(torch.cuda.max_memory_reserved() / 1e9 if _on_cuda() else 0):.1f} GB | "
                 f"loss ekor {tail:.4f} | std prediksi {sd:.3f}")
            info.update(train_bce=float(tot / max(nseen, 1)), tail_loss=float(tail), val_f05=float(f05), val_ftuned=float(ft), pred_std=sd)
        reasons = [info["reason"]] if info["aborted"] else []
        if not info["aborted"]:
            if info["tail_loss"] > collapse_tail: reasons.append(f"loss 25% micro-batch terakhir {info['tail_loss']:.3f} > {collapse_tail}")
            if info["pred_std"] < collapse_std: reasons.append(f"std prediksi {info['pred_std']:.3f} < {collapse_std}")
            if info["val_ftuned"] < collapse_f1: reasons.append(f"F1@tuned {info['val_ftuned']:.3f} < {collapse_f1}")
        if forced(name.split()[-1], att): reasons.append("dipaksa LLM_FORCE_COLLAPSE (hook uji)")
        info.update(collapsed=bool(reasons), reason="; ".join(reasons), minutes=round((time.time() - t_f) / 60, 2))
        return model, info

    def fit_with_retry(name, tri, vnx, s0):
        # percobaan ulang bila kolaps; bila semua kolaps dipakai percobaan dengan loss ekor terendah. return (model, log percobaan, sehat?)
        attempts, best = [], None
        for att in range(n_att):
            model, info = train_one(name, att, tri, vnx, s0 + 7 * att, lr * 0.5 ** att, head_lr * 0.5 ** att)
            attempts.append(info); used = att
            if not info["collapsed"]: break
            key_ = float("inf") if info["aborted"] else info["tail_loss"]
            if att + 1 < n_att:
                if best is None or key_ < best[0]:
                    best = (key_, att, {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if _keep_key(k)})
                tlog(f"  !! {name} percobaan {att} KOLAPS ({info['reason']}) -> latih ulang: seed {s0 + 7 * (att + 1)}, "
                     f"lr {lr * 0.5 ** (att + 1):.1e}, head_lr {head_lr * 0.5 ** (att + 1):.1e}")
                del model; gc.collect()
                if _on_cuda(): torch.cuda.empty_cache()
            else:
                msg = f"  !! PERINGATAN {name}: semua {n_att} percobaan kolaps ({info['reason']})"
                if best is not None and best[0] < key_:
                    model.load_state_dict(best[2], strict=False); used = best[1]; msg += f" -> memakai bobot percobaan {best[1]} (loss ekor terendah)"
                tlog(msg)
        attempts[used]["used"] = True
        return model, attempts, not attempts[used]["collapsed"]

    def dump_meta(final=False):                                    # selama training -> *.partial.json; meta final ditulis SETELAH semua npy
        LLM_TRAIN_META[tag] = meta                                 # (run terputus tidak pernah membuat pasangan meta/npy yang tidak cocok)
        with open(f"{ART}/l1/{tag}_train_meta{'' if final else '.partial'}.json", "w", encoding="utf-8") as fh: json.dump(meta, fh, indent=1)

    def predict_test(model):                                       # (prediksi komentar test, caption-probe test)
        cp = dict(zip(ucap_te, predict_grouped(model, ucap_te, list(range(len(ucap_te))), [[enc.eos]] * len(ucap_te), pre, pad)))
        return predict_grouped(model, cte, list(range(len(test))), com_te, pre, pad), np.array([cp[c] for c in cte], np.float32)

    for f in range(K):
        tri, vai = np.where((FOLD != f) & TRAIN_OK)[0], np.where(FOLD == f)[0]; vnx = vai[~IS_X[vai] & TRAIN_OK[vai]]   # final: latih & validasi hanya baris ber-sentimen
        model, attempts, _ok = fit_with_retry(f"fold {f}", tri, vnx, seed + f)       # percobaan 0 = seeding v7 (seed + fold)
        meta["folds"][str(f)] = attempts
        oof[vai] = predict_grouped(model, ctr, list(vai), com_tr, pre, pad)
        vcaps = sorted(set(ctr[i] for i in vai if ctr[i]))
        if vcaps:
            cmap = dict(zip(vcaps, predict_grouped(model, vcaps, list(range(len(vcaps))), [[enc.eos]] * len(vcaps), pre, pad)))   # caption-probe
            for i in vai:
                if ctr[i]: oof_cap[i] = cmap[ctr[i]]
        p_t, p_c = predict_test(model); tst += p_t / K; tst_cap += p_c / K
        torch.save({k: v.detach().cpu() for k, v in model.state_dict().items() if _keep_key(k)}, f"{ART}/models/{tag}/fold{f}.pth")
        tlog(f"  fold {f} selesai | {len(attempts)} percobaan | {(time.time() - t_all) / 60:.0f} menit sejak mulai")
        del model; gc.collect()
        if _on_cuda(): torch.cuda.empty_cache()
        dump_meta()
    out = {"oof": oof, "test": tst, "oof_cap": oof_cap, "test_cap": tst_cap}
    if full_refit:
        model, attempts, ok = fit_with_retry("full-data", np.arange(len(train)), None, seed + 100)
        meta["full"] = attempts
        if ok:
            out["test_full"], out["test_cap_full"] = predict_test(model)
            torch.save({k: v.detach().cpu() for k, v in model.state_dict().items() if _keep_key(k)}, f"{ART}/models/{tag}/full.pth")
            tlog(f"  full-data selesai | korelasi prediksi test vs rata-rata {K} fold {np.corrcoef(out['test_full'].ravel(), tst.ravel())[0, 1]:.4f} | "
                 f"rata-rata p {out['test_full'].mean():.4f} vs {tst.mean():.4f} | {(time.time() - t_all) / 60:.0f} menit sejak mulai")
        else:
            tlog("  !! full-data kolaps di semua percobaan -> diabaikan (varian refit memakai rata-rata fold saja)")
        del model; gc.collect()
        if _on_cuda(): torch.cuda.empty_cache()
        dump_meta()
    for k, v in out.items(): np.save(f"{ART}/l1/{tag}_{k}.npy", v)
    for k in ["test_full", "test_cap_full"]:                       # refit lama yang tidak terulang tidak boleh ikut termuat dari cache
        if k not in out and os.path.exists(f"{ART}/l1/{tag}_{k}.npy"): os.remove(f"{ART}/l1/{tag}_{k}.npy")
    dump_meta(final=True)
    if os.path.exists(f"{ART}/l1/{tag}_train_meta.partial.json"): os.remove(f"{ART}/l1/{tag}_train_meta.partial.json")
    r = report(tag, oof); RESULTS.append(r)
    n_extra = sum(len(a) - 1 for a in meta["folds"].values()) + (len(meta["full"]) - 1 if "full" in meta else 0)
    tlog(f"[{tag}] OOF(nonX) F1@tuned {r['F1@tuned(OOF)']:.4f} | cv-thr-global {r['F1@cv-thr-global']:.4f} | total {(time.time() - t_all) / 60:.1f} min | "
         f"latih ulang {n_extra}x")
    try:                                                           # salinan log run ini (ditampilkan lagi bila nanti dimuat dari cache)
        lines = open(f"{ART}/train_log.txt", encoding="utf-8").read().split(chr(10))
        i0 = max(i for i, l in enumerate(lines) if l.startswith(f"[{tag}] {model_name} |"))
        open(f"{ART}/l1/{tag}_log.txt", "w", encoding="utf-8").write(chr(10).join(lines[i0:]).rstrip() + chr(10))
    except Exception:
        pass
    return out

### 5.3 Sailor2-3B

In [10]:
run_model("sent_sailor_s1", "llm", seed=SEED + 1000)

[sent_sailor_s1] sudah ada di artifacts_final/l1 -> dimuat, tidak dilatih ulang. Log pelatihan aslinya:
[sent_sailor_s1] sail/Sailor2-3B | QLoRA4bit=False r=16 | lr=0.0002 bs<=16 tok_budget=1024 epochs=1 | komentar 29.8 tok, prefix 98.0 tok/baris | full_refit=False bidir=False
   fold 0 ep 1 micro-batch     0/1427 | loss 1.3307
   fold 0 ep 1 micro-batch   400/1427 | loss 0.5562
   fold 0 ep 1 micro-batch   800/1427 | loss 0.4162
   fold 0 ep 1 micro-batch  1200/1427 | loss 0.3533
  fold 0 | epoch 1/1 | train_bce 0.3280 | val F1@0.5 0.8108 | val F1@tuned 0.8254 | 1563s | peak mem 9.0 GB | loss ekor 0.2064 | std prediksi 0.268
  fold 0 selesai | 1 percobaan | 33 menit sejak mulai
   fold 1 ep 1 micro-batch     0/1484 | loss 0.4936
   fold 1 ep 1 micro-batch   400/1484 | loss 0.6074
  fold 1 | percobaan 0 DIHENTIKAN di micro-batch 741/1484: loss micro-batch 519-741 = 0.494 > 0.42 (level prior)
  !! fold 1 percobaan 0 KOLAPS (loss micro-batch 519-741 = 0.494 > 0.42 (level prior)) -> latih

         model  macroF1 3 kelas  skor resmi (CV bias global)  korelasi vs tfidf
    sent_tfidf           0.5665                       0.6374             1.0000
      sent_ibl           0.6824                       0.6937             0.8622
sent_sailor_s1           0.7289                       0.7362             0.8628


                          blend          bias  skor resmi CV
                 sent_sailor_s1        global         0.7362
                 sent_sailor_s1 per-pilar 50%         0.7349
         sent_sailor_s1+tfidf/2        global         0.7314
         sent_sailor_s1+tfidf/2 per-pilar 50%         0.7290
        sent_ibl+sent_sailor_s1        global         0.7250
sent_ibl+sent_sailor_s1+tfidf/2        global         0.7238
sent_ibl+sent_sailor_s1+tfidf/2 per-pilar 50%         0.7225
        sent_ibl+sent_sailor_s1 per-pilar 50%         0.7219
               sent_ibl+tfidf/2        global         0.6958
               sent_ibl+tfidf/2 per-pilar 50%         0.6957
                       sent_ibl        global         0.6937
                       sent_ibl per-pilar 50%         0.6906



TERPILIH: sent_sailor_s1 | bias global | skor resmi CV (jujur) 0.7362 | in-sample 0.7387
bias [Negatif, Netral, Positif]: [0.0, 1.1, 0.9]


        TidakRelevan  Negatif  Netral  Positif  MACRO
Infra          0.973    0.795   0.444    0.440  0.663
Kes            0.996    0.887   0.540    0.874  0.824
Adm            0.990    0.831   0.656    0.664  0.786
Ling           0.987    0.797   0.371    0.789  0.736
Tertib         0.989    0.864   0.342    0.606  0.700
Trans          0.989    0.801   0.521    0.771  0.770
Sos            0.993    0.877   0.521    0.718  0.777
Keam           0.981    0.781   0.375    0.588  0.681
Pend           0.982    0.831   0.400    0.627  0.710
proporsi kelas OOF (prediksi vs sebenarnya): TidakRelevan 0.8885/0.8913, Negatif 0.0835/0.0835, Netral 0.0132/0.0107, Positif 0.0147/0.0145


submission.csv tersimpan (34687, 10) | proporsi kelas test: TidakRelevan 0.8894, Negatif 0.0789, Netral 0.0173, Positif 0.0144


## 6. Submisi final
Perakitan terakhir dengan semua model yang tersedia; berkas yang dikumpulkan adalah `submission.csv`.

In [11]:
build_submission()
sub = pd.read_csv("submission.csv"); smp = d["sample"]
assert list(sub.columns) == list(smp.columns) and (sub.ID.values == smp.ID.values).all() and set(np.unique(sub[LABELS].values)) <= set(C4)
print("konfigurasi final:", json.load(open("artifacts_final/final_config.json")))
display(sub.head())

         model  macroF1 3 kelas  skor resmi (CV bias global)  korelasi vs tfidf
    sent_tfidf           0.5665                       0.6374             1.0000
      sent_ibl           0.6824                       0.6937             0.8622
sent_sailor_s1           0.7289                       0.7362             0.8628


                          blend          bias  skor resmi CV
                 sent_sailor_s1        global         0.7362
                 sent_sailor_s1 per-pilar 50%         0.7349
         sent_sailor_s1+tfidf/2        global         0.7314
         sent_sailor_s1+tfidf/2 per-pilar 50%         0.7290
        sent_ibl+sent_sailor_s1        global         0.7250
sent_ibl+sent_sailor_s1+tfidf/2        global         0.7238
sent_ibl+sent_sailor_s1+tfidf/2 per-pilar 50%         0.7225
        sent_ibl+sent_sailor_s1 per-pilar 50%         0.7219
               sent_ibl+tfidf/2        global         0.6958
               sent_ibl+tfidf/2 per-pilar 50%         0.6957
                       sent_ibl        global         0.6937
                       sent_ibl per-pilar 50%         0.6906



TERPILIH: sent_sailor_s1 | bias global | skor resmi CV (jujur) 0.7362 | in-sample 0.7387
bias [Negatif, Netral, Positif]: [0.0, 1.1, 0.9]


        TidakRelevan  Negatif  Netral  Positif  MACRO
Infra          0.973    0.795   0.444    0.440  0.663
Kes            0.996    0.887   0.540    0.874  0.824
Adm            0.990    0.831   0.656    0.664  0.786
Ling           0.987    0.797   0.371    0.789  0.736
Tertib         0.989    0.864   0.342    0.606  0.700
Trans          0.989    0.801   0.521    0.771  0.770
Sos            0.993    0.877   0.521    0.718  0.777
Keam           0.981    0.781   0.375    0.588  0.681
Pend           0.982    0.831   0.400    0.627  0.710
proporsi kelas OOF (prediksi vs sebenarnya): TidakRelevan 0.8885/0.8913, Negatif 0.0835/0.0835, Netral 0.0132/0.0107, Positif 0.0147/0.0145


submission.csv tersimpan (34687, 10) | proporsi kelas test: TidakRelevan 0.8894, Negatif 0.0789, Netral 0.0173, Positif 0.0144


konfigurasi final: {'blend': ['sent_sailor_s1'], 'weights': [1.0], 'bias_mode': 'global', 'bias': [0.0, 1.1, 0.9], 'threshold': 0.4, 'cv': 0.7361898024867859}


,ID,Infrastruktur,Kesehatan,Administrasi_Perizinan,Lingkungan_Kebersihan,Ketertiban_Umum,Transportasi_LaluLintas,Sosial_Bantuan,Keamanan_Kriminalitas,Pendidikan
0,1,Negatif,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan
1,2,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,Netral,TidakRelevan,TidakRelevan,TidakRelevan
2,3,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,Netral,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan
3,4,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,Netral
4,5,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan,Negatif,TidakRelevan,TidakRelevan,TidakRelevan,TidakRelevan
